In [1]:
print("hello world")

hello world


In [3]:
from pathlib import Path
import pandas as pd

# --------------------------------------------------
# Project and dataset paths
# --------------------------------------------------

PROJECT_ROOT = Path(r"C:\Users\danie\supportmind-ai")

DATA_PATH = (
    PROJECT_ROOT
    / "rawsupportminddata"
    / "final clean Customer Support Ticket Dataset"
    / "supportmind_final_clean_dataset.csv"
)

print("Dataset path:")
print(DATA_PATH)

print("\nFile exists:", DATA_PATH.exists())

# --------------------------------------------------
# Load dataset
# --------------------------------------------------

df = pd.read_csv(DATA_PATH)

print("\nDataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
# --------------------------------------------------
# 2. Inspect topic-modeling text columns
# --------------------------------------------------
TEXT_COLUMNS = [
    "Ticket Type",
    "Ticket Subject",
    "Ticket Description",
    "Resolution"
]

print("\n" + "=" * 70)
print("TEXT COLUMN ANALYSIS")
print("=" * 70)

for column in TEXT_COLUMNS:
    print(f"\n--- {column} ---")
    print("Missing values:", df[column].isna().sum())
    print("Non-null values:", df[column].notna().sum())
    print("Unique values:", df[column].nunique())

    print("\nSample values:")
    print(df[column].dropna().head(5).to_string(index=False))


# --------------------------------------------------
# 3. Analyze Ticket Type and Ticket Subject distributions
# --------------------------------------------------
print("\n" + "=" * 70)
print("TICKET TYPE DISTRIBUTION")
print("=" * 70)
print(df["Ticket Type"].value_counts(dropna=False))


print("\n" + "=" * 70)
print("TOP 20 TICKET SUBJECTS")
print("=" * 70)
print(df["Ticket Subject"].value_counts(dropna=False).head(20))


# --------------------------------------------------
# 4. Analyze Ticket Description quality
# --------------------------------------------------
descriptions = df["Ticket Description"].dropna().astype(str)

print("\n" + "=" * 70)
print("TICKET DESCRIPTION ANALYSIS")
print("=" * 70)

print("Total non-null descriptions:", len(descriptions))
print("Empty/whitespace descriptions:",
      descriptions.str.strip().eq("").sum())

description_lengths = descriptions.str.len()
word_counts = descriptions.str.split().str.len()

print("\nCharacter length statistics:")
print(description_lengths.describe())

print("\nWord count statistics:")
print(word_counts.describe())


# --------------------------------------------------
# 5. Check exact duplicate descriptions
# --------------------------------------------------
print("\n" + "=" * 70)
print("DUPLICATE ANALYSIS")
print("=" * 70)

print(
    "Exact duplicate Ticket Descriptions:",
    descriptions.duplicated().sum()
)

print(
    "Duplicate rows based on Ticket Subject + Ticket Description:",
    df.duplicated(
        subset=["Ticket Subject", "Ticket Description"]
    ).sum()
)


# --------------------------------------------------
# 6. Inspect random examples
# --------------------------------------------------
print("\n" + "=" * 70)
print("RANDOM TICKET EXAMPLES")
print("=" * 70)

sample_columns = [
    "Ticket Type",
    "Ticket Subject",
    "Ticket Description"
]

print(
    df[sample_columns]
    .sample(10, random_state=42)
    .to_string(index=False)
)

Dataset path:
C:\Users\danie\supportmind-ai\rawsupportminddata\final clean Customer Support Ticket Dataset\supportmind_final_clean_dataset.csv

File exists: True

Dataset shape: (8469, 17)

Columns:
['Ticket ID', 'Customer Name', 'Customer Email', 'Customer Age', 'Customer Gender', 'Product Purchased', 'Date of Purchase', 'Ticket Type', 'Ticket Subject', 'Ticket Description', 'Ticket Status', 'Resolution', 'Ticket Priority', 'Ticket Channel', 'First Response Time', 'Time to Resolution', 'Customer Satisfaction Rating']

TEXT COLUMN ANALYSIS

--- Ticket Type ---
Missing values: 0
Non-null values: 8469
Unique values: 5

Sample values:
Technical issue
Technical issue
Technical issue
Billing inquiry
Billing inquiry

--- Ticket Subject ---
Missing values: 0
Non-null values: 8469
Unique values: 16

Sample values:
           Product setup
Peripheral compatibility
         Network problem
          Account access
               Data loss

--- Ticket Description ---
Missing values: 0
Non-null va

In [4]:
# ============================================================
# EDA STEP 2 — Inspect Ticket Description Structure
# ============================================================

import re
from collections import Counter

text = df["Ticket Description"].astype(str)


# ------------------------------------------------------------
# 1. Common sentence / phrase patterns
# ------------------------------------------------------------
print("=" * 70)
print("COMMON DESCRIPTION PHRASES")
print("=" * 70)

words = re.findall(r"\b[a-zA-Z][a-zA-Z'-]*\b", " ".join(text).lower())

word_counts = Counter(words)

print("\nTop 30 words:")
for word, count in word_counts.most_common(30):
    print(f"{word:<20} {count}")


# ------------------------------------------------------------
# 2. Potential boilerplate phrases
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("COMMON SENTENCE STARTS / PHRASES")
print("=" * 70)

phrase_patterns = [
    r"^I'm having an issue with",
    r"^I'm facing a problem with",
    r"^I'm having trouble",
    r"^I've noticed",
    r"^I've encountered",
    r"^I would like",
    r"^Please",
    r"^I'm concerned",
    r"^Thanks",
    r"^I cannot",
    r"^Will my",
    r"^I've tried",
    r"^I've checked",
]

for pattern in phrase_patterns:
    matches = text.str.contains(
        pattern,
        case=False,
        regex=True,
        na=False
    ).sum()

    print(f"{pattern:<45} {matches}")


# ------------------------------------------------------------
# 3. Placeholder / URL / email / @mention / number signals
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("SPECIAL CONTENT SIGNALS")
print("=" * 70)

patterns = {
    "Placeholder [PLACEHOLDER]": r"\[PLACEHOLDER\]",
    "URLs": r"https?://|www\.",
    "Email addresses": r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
    "@mentions": r"@[A-Za-z0-9_]+",
    "Numbers": r"\b\d+(?:[.,]\d+)?\b",
    "Currency symbols": r"[$€£₹]",
}

for name, pattern in patterns.items():
    matches = text.str.contains(
        pattern,
        case=False,
        regex=True,
        na=False
    ).sum()

    percentage = matches / len(text) * 100

    print(f"{name:<30} {matches:>5} ({percentage:>6.2f}%)")


# ------------------------------------------------------------
# 4. Very short / very long descriptions
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("EXTREME LENGTH EXAMPLES")
print("=" * 70)

word_count = text.str.split().str.len()

print("\nShortest 10 descriptions:")
print(
    df.loc[word_count.nsmallest(10).index,
           ["Ticket Subject", "Ticket Description"]]
    .to_string(index=False)
)

print("\nLongest 10 descriptions:")
print(
    df.loc[word_count.nlargest(10).index,
           ["Ticket Subject", "Ticket Description"]]
    .to_string(index=False)
)


# ------------------------------------------------------------
# 5. Compare Subject vs Description vocabulary
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("SUBJECT / DESCRIPTION VOCABULARY")
print("=" * 70)

subject_words = set(
    re.findall(
        r"\b[a-zA-Z][a-zA-Z'-]*\b",
        " ".join(df["Ticket Subject"].astype(str)).lower()
    )
)

description_words = set(words)

overlap = subject_words & description_words

print("Unique subject words:", len(subject_words))
print("Unique description words:", len(description_words))
print("Vocabulary overlap:", len(overlap))

print("\nSubject vocabulary:")
print(sorted(subject_words))


# ------------------------------------------------------------
# 6. Subject + description examples
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("SUBJECT vs DESCRIPTION — 20 RANDOM EXAMPLES")
print("=" * 70)

print(
    df[["Ticket Subject", "Ticket Description"]]
    .sample(20, random_state=123)
    .to_string(index=False)
)

COMMON DESCRIPTION PHRASES

Top 30 words:
the                  26430
issue                11527
to                   10806
i'm                  10292
my                   9591
please               8815
with                 8326
it                   7460
an                   7251
a                    6850
having               6430
assist               6258
and                  6020
i've                 6001
i                    5991
is                   5666
you                  4446
but                  4384
this                 4354
on                   3922
in                   3342
for                  3150
of                   2750
be                   2736
that                 2579
problem              2284
not                  2282
can                  2271
if                   2263
product              2180

COMMON SENTENCE STARTS / PHRASES
^I'm having an issue with                     5868
^I'm facing a problem with                    179
^I'm having trouble                    

In [5]:
# ============================================================
# EDA STEP 3 — Repetition, Boilerplate & Subject Consistency
# ============================================================

import re
from collections import Counter

text = df["Ticket Description"].astype(str)
subjects = df["Ticket Subject"].astype(str)


# ------------------------------------------------------------
# 1. Measure exact repetition of common boilerplate sentences
# ------------------------------------------------------------
print("=" * 70)
print("REPEATED BOILERPLATE / SENTENCE PATTERNS")
print("=" * 70)

patterns = {
    "Generic opening":
        r"I'm having an issue with .*?\. Please assist\.",

    "Generic opening - facing":
        r"I'm facing a problem with .*?\. Please assist\.",

    "Generic opening - trouble":
        r"I'm having trouble .*?\.",

    "Unresolved support":
        r"I've already contacted customer support multiple times, but the issue remains unresolved\.",

    "Firmware update pattern":
        r"I've recently updated the firmware.*?issue started happening afterward.*?\?",

    "Troubleshooting pattern":
        r"I've (?:tried|followed|checked).*?(?:issue persists|problem persists|no luck).*?\."
}

for name, pattern in patterns.items():
    matches = text.str.contains(
        pattern,
        case=False,
        regex=True,
        na=False
    ).sum()

    print(f"{name:<35} {matches:>5} ({matches / len(text) * 100:>6.2f}%)")


# ------------------------------------------------------------
# 2. Count how many sentences each description contains
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("SENTENCE COUNT")
print("=" * 70)

def sentence_count(value: str) -> int:
    # Simple sentence boundary estimate for EDA only.
    sentences = re.split(r"(?<=[.!?])\s+", value.strip())
    return len([s for s in sentences if s.strip()])


sentence_counts = text.apply(sentence_count)

print(sentence_counts.describe())

print("\nSentence count distribution:")
print(sentence_counts.value_counts().sort_index())


# ------------------------------------------------------------
# 3. Check how much of each description is repeated boilerplate
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("BOILERPLATE DENSITY")
print("=" * 70)

boilerplate_patterns = [
    r"I'm having an issue with .*?\. Please assist\.",
    r"I'm facing a problem with .*?\. Please assist\.",
    r"I'm having trouble .*?\.",
    r"I've already contacted customer support multiple times, but the issue remains unresolved\.",
    r"I've recently updated the firmware.*?\?",
    r"I've tried .*?\.?",
    r"I've checked .*?\.?",
]

def boilerplate_length(value: str) -> int:
    total = 0

    for pattern in boilerplate_patterns:
        matches = re.findall(
            pattern,
            value,
            flags=re.IGNORECASE
        )

        total += sum(len(match) for match in matches)

    return total


boilerplate_chars = text.apply(boilerplate_length)

boilerplate_ratio = (
    boilerplate_chars / text.str.len()
).clip(upper=1.0)

print("Mean estimated boilerplate ratio:",
      round(boilerplate_ratio.mean(), 4))

print("Median estimated boilerplate ratio:",
      round(boilerplate_ratio.median(), 4))

print("Tickets with >25% estimated boilerplate:",
      (boilerplate_ratio > 0.25).sum())

print("Tickets with >50% estimated boilerplate:",
      (boilerplate_ratio > 0.50).sum())


# ------------------------------------------------------------
# 4. Compare Subject words against Description
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("SUBJECT-DESCRIPTION WORD OVERLAP")
print("=" * 70)

def normalize_words(value: str) -> set[str]:
    return set(
        re.findall(
            r"\b[a-zA-Z][a-zA-Z'-]*\b",
            value.lower()
        )
    )

overlap_ratios = []

for subject, description in zip(subjects, text):

    subject_words = normalize_words(subject)
    description_words = normalize_words(description)

    if subject_words:
        overlap = len(subject_words & description_words)
        ratio = overlap / len(subject_words)
    else:
        ratio = 0

    overlap_ratios.append(ratio)

overlap_ratios = pd.Series(overlap_ratios)

print(overlap_ratios.describe())

print("\nOverlap categories:")

print(
    "0% overlap:",
    (overlap_ratios == 0).sum()
)

print(
    "1–49% overlap:",
    ((overlap_ratios > 0) & (overlap_ratios < 1)).sum()
)

print(
    "100% overlap:",
    (overlap_ratios == 1).sum()
)


# ------------------------------------------------------------
# 5. Show examples with very low subject-description overlap
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("LOW SUBJECT-DESCRIPTION OVERLAP EXAMPLES")
print("=" * 70)

low_overlap_indices = overlap_ratios.nsmallest(10).index

print(
    df.loc[
        low_overlap_indices,
        ["Ticket Subject", "Ticket Description"]
    ].to_string(index=False)
)


# ------------------------------------------------------------
# 6. Show examples with high subject-description overlap
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("HIGH SUBJECT-DESCRIPTION OVERLAP EXAMPLES")
print("=" * 70)

high_overlap_indices = overlap_ratios.nlargest(10).index

print(
    df.loc[
        high_overlap_indices,
        ["Ticket Subject", "Ticket Description"]
    ].to_string(index=False)
)

REPEATED BOILERPLATE / SENTENCE PATTERNS
Generic opening                      5868 ( 69.29%)
Generic opening - facing                0 (  0.00%)
Generic opening - trouble             178 (  2.10%)
Unresolved support                    460 (  5.43%)
Firmware update pattern               485 (  5.73%)
Troubleshooting pattern              1404 ( 16.58%)

SENTENCE COUNT
count    8469.000000
mean        4.609635
std         1.163908
min         2.000000
25%         4.000000
50%         4.000000
75%         5.000000
max        14.000000
Name: Ticket Description, dtype: float64

Sentence count distribution:
Ticket Description
2        1
3     1350
4     2950
5     2540
6     1115
7      370
8      104
9       24
10      10
11       3
12       1
14       1
Name: count, dtype: int64

BOILERPLATE DENSITY
Mean estimated boilerplate ratio: 0.2229
Median estimated boilerplate ratio: 0.2134
Tickets with >25% estimated boilerplate: 2781
Tickets with >50% estimated boilerplate: 675

SUBJECT-DESCRIPTIO

In [6]:
# ============================================================
# EDA STEP 4 — Sentence-Level Repetition Analysis
# ============================================================

import re
from collections import Counter

text = df["Ticket Description"].astype(str)


# ------------------------------------------------------------
# 1. Split descriptions into sentences
# ------------------------------------------------------------
def split_sentences(value: str) -> list[str]:
    sentences = re.split(r"(?<=[.!?])\s+", value.strip())
    return [s.strip() for s in sentences if s.strip()]


all_sentences = []

for value in text:
    all_sentences.extend(split_sentences(value))


print("=" * 70)
print("SENTENCE-LEVEL CORPUS")
print("=" * 70)

print("Total sentences:", len(all_sentences))
print("Unique sentences:", len(set(all_sentences)))


# ------------------------------------------------------------
# 2. Exact repeated sentences
# ------------------------------------------------------------
sentence_counts = Counter(all_sentences)

print("\n" + "=" * 70)
print("TOP 40 EXACT REPEATED SENTENCES")
print("=" * 70)

for sentence, count in sentence_counts.most_common(40):
    if count > 1:
        print(f"\nCount: {count}")
        print(sentence)


# ------------------------------------------------------------
# 3. Sentence frequency distribution
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("SENTENCE REPETITION DISTRIBUTION")
print("=" * 70)

frequency_counts = Counter(sentence_counts.values())

for frequency in sorted(frequency_counts):
    print(
        f"Sentences appearing {frequency} time(s): "
        f"{frequency_counts[frequency]}"
    )


# ------------------------------------------------------------
# 4. Normalize sentences to identify templates
# ------------------------------------------------------------
def normalize_sentence(sentence: str) -> str:

    value = sentence.lower().strip()

    # Replace product-like names with placeholder for template detection
    value = re.sub(
        r"\b(?:gopro hero|lg smart tv|microsoft office|"
        r"sony playstation|iphone|xbox|amazon echo|"
        r"nintendo switch|nintendo switch pro controller|"
        r"roomba robot vacuum|google nest|"
        r"fitbit versa smartwatch|apple airpods)\b",
        "<PRODUCT>",
        value,
        flags=re.IGNORECASE
    )

    # Normalize numbers
    value = re.sub(r"\b\d+(?:[.,]\d+)?\b", "<NUMBER>", value)

    # Normalize whitespace
    value = re.sub(r"\s+", " ", value)

    return value


normalized_sentences = [
    normalize_sentence(sentence)
    for sentence in all_sentences
]

template_counts = Counter(normalized_sentences)


# ------------------------------------------------------------
# 5. Most common normalized templates
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("TOP 40 NORMALIZED SENTENCE TEMPLATES")
print("=" * 70)

for template, count in template_counts.most_common(40):

    if count > 3:

        print(f"\nCount: {count}")
        print(template)


# ------------------------------------------------------------
# 6. How concentrated is the sentence corpus?
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("SENTENCE CORPUS CONCENTRATION")
print("=" * 70)

for threshold in [2, 5, 10, 25, 50, 100]:

    count = sum(
        1
        for frequency in template_counts.values()
        if frequency >= threshold
    )

    print(
        f"Templates appearing >= {threshold:>3} times: "
        f"{count}"
    )


# ------------------------------------------------------------
# 7. Show a few complete descriptions containing
#    highly repeated non-opening sentences
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("EXAMPLES WITH REPEATED SENTENCE TEMPLATES")
print("=" * 70)

repeated_templates = {
    template
    for template, count in template_counts.items()
    if count >= 25
}

shown = 0

for index, value in enumerate(text):

    sentences = split_sentences(value)

    matched = [
        sentence
        for sentence in sentences
        if normalize_sentence(sentence) in repeated_templates
    ]

    if matched:

        print(f"\n--- Ticket row {index} ---")
        print("Subject:", df.loc[index, "Ticket Subject"])
        print("Description:", value)

        shown += 1

        if shown >= 10:
            break

SENTENCE-LEVEL CORPUS
Total sentences: 39039
Unique sentences: 14772

TOP 40 EXACT REPEATED SENTENCES

Count: 6182
Please assist.

Count: 498
Sometimes it works fine, but other times it acts up unexpectedly.

Count: 490
Could you please guide me through the steps?

Count: 485
Could it be related to the update?

Count: 472
It says '[PLACEHOLDER]'.

Count: 472
What does it mean?

Count: 470
I haven't made any other changes to the device.

Count: 286
It used to last much longer.

Count: 224
Thank you.

Count: 189
It keeps displaying an 'Invalid Credentials' error, even though I'm using the correct login information.

Count: 189
How can I regain access to my account?

Count: 188
How can I recover my account?

Count: 186
How can I troubleshoot this network problem?

Count: 184
It keeps disconnecting intermittently.

Count: 183
Whenever I try to perform a specific action, the application crashes.

Count: 183
Are there any updates or fixes available?

Count: 182
I suspect there might be a har

In [7]:
# ============================================================
# EDA STEP 5 — Estimate Removable Boilerplate Per Ticket
# ============================================================

import re

# Clearly generic support/conversational sentences identified
# from the repetition analysis.
generic_sentences = {
    "please assist.",
    "thank you.",
    "what should i do?",
    "can you please help me with this?",
    "can you please provide a solution?",
    "how can i resolve this issue?",
    "what does it mean?",
    "could you please guide me through the steps?",
    "what can be done to resolve this issue?",
    "how can i troubleshoot this network problem?",
    "what steps should i take to troubleshoot this issue?",
    "are there any updates or fixes available?",
    "how can i regain access to my account?",
    "how can i recover my account?",
    "can you guide me on how to retrieve them?",
    "i need them urgently.",
}

def split_sentences(value: str) -> list[str]:
    sentences = re.split(r"(?<=[.!?])\s+", value.strip())
    return [s.strip() for s in sentences if s.strip()]

def normalize_for_matching(sentence: str) -> str:
    return re.sub(r"\s+", " ", sentence.lower().strip())

results = []

for index, value in enumerate(df["Ticket Description"].astype(str)):

    sentences = split_sentences(value)

    removable = []
    informative = []

    for sentence in sentences:

        normalized = normalize_for_matching(sentence)

        if normalized in generic_sentences:
            removable.append(sentence)
        else:
            informative.append(sentence)

    total = len(sentences)
    removable_count = len(removable)

    results.append({
        "row": index,
        "total_sentences": total,
        "removable_sentences": removable_count,
        "remaining_sentences": len(informative),
        "removable_ratio": removable_count / total if total else 0,
    })

boilerplate_df = pd.DataFrame(results)

print("=" * 70)
print("TARGETED BOILERPLATE ANALYSIS")
print("=" * 70)

print(
    "\nAverage removable sentence ratio:",
    round(boilerplate_df["removable_ratio"].mean(), 4)
)

print(
    "Median removable sentence ratio:",
    round(boilerplate_df["removable_ratio"].median(), 4)
)

print(
    "Tickets with >= 25% removable:",
    int((boilerplate_df["removable_ratio"] >= 0.25).sum())
)

print(
    "Tickets with >= 50% removable:",
    int((boilerplate_df["removable_ratio"] >= 0.50).sum())
)

print(
    "Tickets with >= 75% removable:",
    int((boilerplate_df["removable_ratio"] >= 0.75).sum())
)

print(
    "Tickets with ALL sentences removable:",
    int((boilerplate_df["remaining_sentences"] == 0).sum())
)

print("\n" + "=" * 70)
print("REMAINING SENTENCE COUNT")
print("=" * 70)

print(
    boilerplate_df["remaining_sentences"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\n" + "=" * 70)
print("HIGHEST BOILERPLATE TICKETS")
print("=" * 70)

print(
    boilerplate_df
    .sort_values("removable_ratio", ascending=False)
    .head(20)
    .to_string(index=False)
)

TARGETED BOILERPLATE ANALYSIS

Average removable sentence ratio: 0.2485
Median removable sentence ratio: 0.25
Tickets with >= 25% removable: 5229
Tickets with >= 50% removable: 174
Tickets with >= 75% removable: 3
Tickets with ALL sentences removable: 0

REMAINING SENTENCE COUNT
remaining_sentences
2     1449
3     3137
4     2565
5     1018
6      244
7       40
8       11
9        3
10       1
11       1

HIGHEST BOILERPLATE TICKETS
 row  total_sentences  removable_sentences  remaining_sentences  removable_ratio
3087               14                   12                    2         0.857143
4556               10                    8                    2         0.800000
7482                9                    7                    2         0.777778
6208               10                    7                    3         0.700000
3521                8                    5                    3         0.625000
4926                5                    3                    2         0.6

In [8]:
# ============================================================
# EDA STEP 6 — Sentence Relevance / Topic Consistency
# ============================================================

import re
import pandas as pd
from collections import Counter

def split_sentences(value: str) -> list[str]:
    sentences = re.split(r"(?<=[.!?])\s+", value.strip())
    return [s.strip() for s in sentences if s.strip()]

# Extract sentences from every ticket
ticket_sentences = []

for index, value in enumerate(df["Ticket Description"].astype(str)):
    sentences = split_sentences(value)

    ticket_sentences.append({
        "row": index,
        "sentences": sentences
    })

# ------------------------------------------------------------
# 1. Inspect suspiciously unrelated sentences manually
# ------------------------------------------------------------

print("=" * 70)
print("SUSPICIOUS SENTENCE EXAMPLES")
print("=" * 70)

# Keywords seen in the contamination examples.
# These are ONLY for investigation, not cleaning rules.
suspicious_patterns = [
    r"\bbilling zip code\b",
    r"\bwebsite address\b",
    r"\bfeedback\b",
    r"\bseller\b",
    r"\bdamages arising\b",
    r"\bbattleground game\b",
    r"\bsuppliers\b",
    r"\binventory\b",
    r"\brequest\b.*\bdav\b",
]

pattern = re.compile(
    "|".join(suspicious_patterns),
    flags=re.IGNORECASE
)

found = []

for item in ticket_sentences:

    for sentence in item["sentences"]:

        if pattern.search(sentence):

            found.append({
                "row": item["row"],
                "sentence": sentence
            })

print("Suspicious sentence matches:", len(found))

for item in found[:40]:

    print(f"\nRow: {item['row']}")
    print("Sentence:", item["sentence"])


# ------------------------------------------------------------
# 2. Count sentences containing suspicious patterns
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SUSPICIOUS SENTENCE FREQUENCY")
print("=" * 70)

sentence_counter = Counter(
    item["sentence"]
    for item in found
)

for sentence, count in sentence_counter.most_common(30):

    print(f"\nCount: {count}")
    print(sentence)


# ------------------------------------------------------------
# 3. Inspect complete tickets containing suspicious content
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("COMPLETE TICKET EXAMPLES")
print("=" * 70)

shown_rows = set()

for item in found:

    row = item["row"]

    if row in shown_rows:
        continue

    shown_rows.add(row)

    print(f"\n--- Ticket row {row} ---")
    print("Subject:", df.loc[row, "Ticket Subject"])
    print("Description:", df.loc[row, "Ticket Description"])

    if len(shown_rows) >= 15:
        break

SUSPICIOUS SENTENCE EXAMPLES
Suspicious sentence matches: 177

Row: 0
Sentence: Your billing zip code is: 71701.

Row: 0
Sentence: We appreciate that you have requested a website address.

Row: 3
Sentence: If you have a problem you're interested in and I'd love to see this happen, please check out the Feedback.

Row: 4
Sentence: Note: The seller is not responsible for any damages arising out of the delivery of the battleground game.

Row: 7
Sentence: (Thanks) I will contact all my suppliers and confirm.

Row: 7
Sentence: Please try and find out whether their inventory is currently stocked, or any other reason.

Row: 9
Sentence: If we can, please send a "request" to dav The issue I'm facing is intermittent.

Row: 10
Sentence: Report Feedback Customer Service is your best I'm using the original charger that came with my Nintendo Switch, but it's not charging properly.

Row: 31
Sentence: The seller has sold the item for $29.89.

Row: 147
Sentence: Thank you for your information." It has b

In [9]:
# ============================================================
# PREPROCESSING RULE 1
# Remove generic conversational/support boilerplate
# ============================================================

import re
import pandas as pd

GENERIC_BOILERPLATE = {
    "please assist.",
    "thank you.",
    "what should i do?",
    "can you please help me with this?",
    "can you please provide a solution?",
    "how can i resolve this issue?",
    "what does it mean?",
    "could you please guide me through the steps?",
    "what can be done to resolve this issue?",
    "how can i troubleshoot this network problem?",
    "what steps should i take to troubleshoot this issue?",
    "are there any updates or fixes available?",
    "how can i regain access to my account?",
    "how can i recover my account?",
    "can you guide me on how to retrieve them?",
    "i need them urgently.",
}


def split_sentences(text: str) -> list[str]:
    """Split a ticket description into sentences."""
    
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    
    return [
        sentence.strip()
        for sentence in sentences
        if sentence.strip()
    ]


def normalize_sentence(sentence: str) -> str:
    """Normalize only for matching boilerplate."""
    
    return re.sub(
        r"\s+",
        " ",
        sentence.lower().strip()
    )


def remove_generic_boilerplate(text: str) -> str:
    """
    Remove only explicitly identified generic support sentences.

    No product names, numbers, technical terms, or issue-bearing
    sentences are removed by this rule.
    """
    
    sentences = split_sentences(text)
    
    remaining_sentences = [
        sentence
        for sentence in sentences
        if normalize_sentence(sentence) not in GENERIC_BOILERPLATE
    ]
    
    return " ".join(remaining_sentences)


# Apply Rule 1
df["description_rule1"] = (
    df["Ticket Description"]
    .astype(str)
    .apply(remove_generic_boilerplate)
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

df["original_sentence_count"] = (
    df["Ticket Description"]
    .astype(str)
    .apply(lambda x: len(split_sentences(x)))
)

df["rule1_sentence_count"] = (
    df["description_rule1"]
    .apply(lambda x: len(split_sentences(x)))
)

df["removed_sentence_count"] = (
    df["original_sentence_count"]
    - df["rule1_sentence_count"]
)


print("=" * 70)
print("RULE 1 — PREPROCESSING VALIDATION")
print("=" * 70)

print(
    "\nOriginal tickets:",
    len(df)
)

print(
    "Empty after Rule 1:",
    int((df["description_rule1"].str.strip() == "").sum())
)

print(
    "Tickets with at least one sentence removed:",
    int((df["removed_sentence_count"] > 0).sum())
)

print(
    "Total sentences removed:",
    int(df["removed_sentence_count"].sum())
)

print(
    "Average sentences removed per ticket:",
    round(df["removed_sentence_count"].mean(), 4)
)

print(
    "Maximum sentences removed from one ticket:",
    int(df["removed_sentence_count"].max())
)


print("\n" + "=" * 70)
print("REMOVAL DISTRIBUTION")
print("=" * 70)

print(
    df["removed_sentence_count"]
    .value_counts()
    .sort_index()
    .to_string()
)


# ------------------------------------------------------------
# Before / after examples
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BEFORE / AFTER EXAMPLES")
print("=" * 70)

examples = df[
    df["removed_sentence_count"] > 0
].head(10)

for index, row in examples.iterrows():

    print(f"\n--- Ticket row {index} ---")

    print("\nBEFORE:")
    print(row["Ticket Description"])

    print("\nAFTER RULE 1:")
    print(row["description_rule1"])

RULE 1 — PREPROCESSING VALIDATION

Original tickets: 8469
Empty after Rule 1: 0
Tickets with at least one sentence removed: 8022
Total sentences removed: 9500
Average sentences removed per ticket: 1.1217
Maximum sentences removed from one ticket: 12

REMOVAL DISTRIBUTION
removed_sentence_count
0      447
1     6671
2     1266
3       65
4       13
5        2
6        1
7        2
8        1
12       1

BEFORE / AFTER EXAMPLES

--- Ticket row 0 ---

BEFORE:
I'm having an issue with the GoPro Hero. Please assist. Your billing zip code is: 71701. We appreciate that you have requested a website address. Please double check your email address. I've tried troubleshooting steps mentioned in the user manual, but the issue persists.

AFTER RULE 1:
I'm having an issue with the GoPro Hero. Your billing zip code is: 71701. We appreciate that you have requested a website address. Please double check your email address. I've tried troubleshooting steps mentioned in the user manual, but the issue per

In [10]:
# ============================================================
# PREPROCESSING RULE 2 — CANDIDATE CONTAMINATION DETECTION
# ============================================================
#
# IMPORTANT:
# This cell DOES NOT modify the dataset.
#
# It identifies candidate unrelated/injected sentences so that
# we can validate the rule before actually removing anything.
# ============================================================

import re
import pandas as pd


def split_sentences(text: str) -> list[str]:
    """Split a ticket description into sentences."""

    sentences = re.split(
        r"(?<=[.!?])\s+",
        text.strip()
    )

    return [
        sentence.strip()
        for sentence in sentences
        if sentence.strip()
    ]


# ------------------------------------------------------------
# Candidate contamination patterns
# ------------------------------------------------------------
#
# These patterns are deliberately conservative.
# They target structures observed in our EDA rather than
# generic words such as "seller", "product", or "feedback".
#
# They are NOT yet final deletion rules.
# ------------------------------------------------------------

CONTAMINATION_PATTERNS = [

    # Marketplace / seller contamination
    (
        "seller_transaction",
        re.compile(
            r"\b(?:seller|sellers)\b.*"
            r"\b(?:sold|selling|merchandise|inventory|"
            r"purchase|purchased|invoice|transaction)\b",
            re.IGNORECASE
        )
    ),

    # Seller + website / product listing contamination
    (
        "seller_listing",
        re.compile(
            r"\b(?:seller|sellers)\b.*"
            r"\b(?:web\s*site|website|product description|"
            r"listed|listing)\b",
            re.IGNORECASE
        )
    ),

    # Supplier / inventory contamination
    (
        "supplier_inventory",
        re.compile(
            r"\b(?:suppliers?|inventory|stocked)\b",
            re.IGNORECASE
        )
    ),

    # Shipping / merchandise contamination
    (
        "shipping_merchandise",
        re.compile(
            r"\b(?:delivery|shipped|shipping|merchandise|"
            r"battleground game)\b.*"
            r"\b(?:seller|item|game|product)\b",
            re.IGNORECASE
        )
    ),

    # Feedback / website fragments
    (
        "feedback_website",
        re.compile(
            r"\b(?:feedback|website address|search engine)\b",
            re.IGNORECASE
        )
    ),

    # Embedded source-code / log contamination
    (
        "source_code_or_log",
        re.compile(
            r"(?:"
            r"\bERROR\b.*"
            r"\[.*\]"
            r"|"
            r"/\*.*\*/"
            r"|"
            r"\b(?:errno|ItemStack|FML)\b"
            r")",
            re.IGNORECASE
        )
    ),

    # Explicitly malformed injected fragments
    (
        "malformed_fragment",
        re.compile(
            r"(?:"
            r"\bReport Feedback Customer Service\b"
            r"|"
            r"\bProduct Search:\s*"
            r"|"
            r"\bplease send a ['\"]request['\"] to\b"
            r"|"
            r"\bretain seller info\b"
            r")",
            re.IGNORECASE
        )
    ),
]


# ------------------------------------------------------------
# Scan the ORIGINAL descriptions
# ------------------------------------------------------------

candidate_rows = []

for index, value in enumerate(
    df["Ticket Description"].astype(str)
):

    sentences = split_sentences(value)

    for sentence in sentences:

        matched_rules = []

        for rule_name, pattern in CONTAMINATION_PATTERNS:

            if pattern.search(sentence):
                matched_rules.append(rule_name)

        if matched_rules:

            candidate_rows.append({
                "row": index,
                "subject": df.loc[index, "Ticket Subject"],
                "sentence": sentence,
                "matched_rules": ", ".join(matched_rules),
            })


candidates_df = pd.DataFrame(candidate_rows)


# ------------------------------------------------------------
# Validation summary
# ------------------------------------------------------------

print("=" * 70)
print("RULE 2 — CANDIDATE CONTAMINATION ANALYSIS")
print("=" * 70)

print(
    "\nCandidate contaminated sentences:",
    len(candidates_df)
)

print(
    "Unique tickets affected:",
    candidates_df["row"].nunique()
    if not candidates_df.empty
    else 0
)

print(
    "Unique candidate sentences:",
    candidates_df["sentence"].nunique()
    if not candidates_df.empty
    else 0
)


# ------------------------------------------------------------
# Rule frequency
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MATCHES BY CONTAMINATION PATTERN")
print("=" * 70)

if not candidates_df.empty:

    rule_counts = (
        candidates_df["matched_rules"]
        .str.split(", ")
        .explode()
        .value_counts()
    )

    print(rule_counts.to_string())

else:
    print("No candidate sentences detected.")


# ------------------------------------------------------------
# Examples
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CANDIDATE SENTENCE EXAMPLES")
print("=" * 70)

if not candidates_df.empty:

    for _, row in candidates_df.head(40).iterrows():

        print(f"\nRow: {row['row']}")
        print(f"Subject: {row['subject']}")
        print(f"Rule: {row['matched_rules']}")
        print(f"Sentence: {row['sentence']}")


# ------------------------------------------------------------
# Complete ticket examples
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("COMPLETE TICKETS WITH CANDIDATE CONTAMINATION")
print("=" * 70)

shown = set()

if not candidates_df.empty:

    for _, candidate in candidates_df.iterrows():

        row = candidate["row"]

        if row in shown:
            continue

        shown.add(row)

        print(f"\n--- Ticket row {row} ---")
        print("Subject:", df.loc[row, "Ticket Subject"])
        print("Description:", df.loc[row, "Ticket Description"])

        if len(shown) >= 15:
            break

RULE 2 — CANDIDATE CONTAMINATION ANALYSIS

Candidate contaminated sentences: 150
Unique tickets affected: 142
Unique candidate sentences: 147

MATCHES BY CONTAMINATION PATTERN
matched_rules
supplier_inventory      52
feedback_website        50
seller_transaction      15
source_code_or_log      15
shipping_merchandise    13
seller_listing           5
malformed_fragment       4

CANDIDATE SENTENCE EXAMPLES

Row: 0
Subject: Product setup
Rule: feedback_website
Sentence: We appreciate that you have requested a website address.

Row: 3
Subject: Account access
Rule: feedback_website
Sentence: If you have a problem you're interested in and I'd love to see this happen, please check out the Feedback.

Row: 4
Subject: Data loss
Rule: shipping_merchandise
Sentence: Note: The seller is not responsible for any damages arising out of the delivery of the battleground game.

Row: 7
Subject: Battery life
Rule: supplier_inventory
Sentence: (Thanks) I will contact all my suppliers and confirm.

Row: 7
Su

In [11]:
# ============================================================
# PREPROCESSING RULE 2
# Remove demonstrably unrelated injected content
# ============================================================
#
# Input:
#     description_rule1
#
# Output:
#     description_rule2
#
# Rule 2 operates conservatively at the fragment level.
# It does NOT use Ticket Subject or Ticket Type.
# ============================================================

import re
import pandas as pd


def clean_injected_content(text: str) -> str:
    """
    Remove clearly unrelated injected fragments while preserving
    genuine support-ticket information.
    """

    value = str(text)

    # --------------------------------------------------------
    # 1. Remove embedded source-code / Minecraft log fragments
    # --------------------------------------------------------

    value = re.sub(
        r"\[[0-9]{1,2}:\d{2}:\d{2}(?:\.\d+)?\]\s*"
        r"\[[^\]]+\]\s*"
        r"\[[^\]]+\]\s*"
        r".*?(?=(?:I['’]?m|I've|I am|The issue|"
        r"This problem|I've noticed|I'm experiencing|"
        r"I've tried|I need|I suspect|I rely|"
        r"I['’]?ve performed|$))",
        "",
        value,
        flags=re.IGNORECASE,
    )

    # Common source-code fragments
    value = re.sub(
        r"/\*.*?\*/",
        "",
        value,
        flags=re.DOTALL,
    )

    value = re.sub(
        r"\b(?:errno\s*=\s*0|"
        r"net\.minecraft\.inventory\.ItemStack|"
        r"\[FML/?\])",
        "",
        value,
        flags=re.IGNORECASE,
    )

    # --------------------------------------------------------
    # 2. Remove explicit malformed injected fragments
    # --------------------------------------------------------

    value = re.sub(
        r"\bProduct Search:\s*What's New in 2-3-4-5\?",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bReport Feedback Customer Service is your best\b",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bplease send a ['\"]request['\"] to\s+\w+",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bRetain Seller Info\b",
        "",
        value,
        flags=re.IGNORECASE,
    )

    # --------------------------------------------------------
    # 3. Remove clearly unrelated seller / marketplace fragments
    # --------------------------------------------------------

    value = re.sub(
        r"\bThe seller has sold the item for \$[\d,.]+\.",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bThe seller is not responsible for any damages "
        r"arising out of the delivery of the battleground game\.",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bPlease have the game in good condition and shipped to you\b",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bFirst, look at the seller['’]?s web site\.",
        "",
        value,
        flags=re.IGNORECASE,
    )

    # --------------------------------------------------------
    # 4. Remove clearly unrelated supplier / inventory fragments
    # --------------------------------------------------------

    value = re.sub(
        r"\(?Thanks\)? I will contact all my suppliers and confirm\.",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bPlease try and find out whether their inventory is "
        r"currently stocked, or any other reason\.",
        "",
        value,
        flags=re.IGNORECASE,
    )

    # --------------------------------------------------------
    # 5. Remove isolated feedback / website injection
    # --------------------------------------------------------

    value = re.sub(
        r"\bWe appreciate that you have requested a website address\.",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bIf you have a problem you're interested in and I'd love "
        r"to see this happen, please check out the Feedback\.",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bIf you have any further feedback, please drop me an "
        r"email at [A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\.",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bPlease send feedback here -V\.3\.0\b",
        "",
        value,
        flags=re.IGNORECASE,
    )

    # --------------------------------------------------------
    # 6. Remove clearly unrelated supplier/seller fragments
    # --------------------------------------------------------

    value = re.sub(
        r"\bWhen we are the only supplier to provide this service\b",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bIf we can't keep this up, and the supplier can't figure "
        r"this out, we should have sent something in this particular "
        r"order right away\.",
        "",
        value,
        flags=re.IGNORECASE,
    )

    # --------------------------------------------------------
    # 7. Remove obvious standalone transaction fragments
    # --------------------------------------------------------

    value = re.sub(
        r"\bYour billing zip code is:\s*\d+\.",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bThe seller says on \d{1,2}/\d{1,2}/\d{2,4}\b",
        "",
        value,
        flags=re.IGNORECASE,
    )

    # --------------------------------------------------------
    # 8. Clean whitespace left by removals
    # --------------------------------------------------------

    value = re.sub(r"\s{2,}", " ", value)
    value = re.sub(r"\s+([,.!?])", r"\1", value)

    return value.strip()


# ------------------------------------------------------------
# Apply Rule 2
# ------------------------------------------------------------

df["description_rule2"] = (
    df["description_rule1"]
    .astype(str)
    .apply(clean_injected_content)
)


# ------------------------------------------------------------
# Validation statistics
# ------------------------------------------------------------

df["rule1_char_count"] = df["description_rule1"].str.len()
df["rule2_char_count"] = df["description_rule2"].str.len()

df["rule2_removed_chars"] = (
    df["rule1_char_count"]
    - df["rule2_char_count"]
)

df["rule2_empty"] = (
    df["description_rule2"].str.strip() == ""
)


print("=" * 70)
print("RULE 2 — PREPROCESSING VALIDATION")
print("=" * 70)

print(
    "\nOriginal tickets:",
    len(df)
)

print(
    "Empty after Rule 2:",
    int(df["rule2_empty"].sum())
)

print(
    "Tickets modified by Rule 2:",
    int((df["rule2_removed_chars"] > 0).sum())
)

print(
    "Total characters removed:",
    int(df["rule2_removed_chars"].sum())
)

print(
    "Average characters removed per ticket:",
    round(df["rule2_removed_chars"].mean(), 4)
)


# ------------------------------------------------------------
# Before / after examples
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("RULE 2 BEFORE / AFTER")
print("=" * 70)

modified = df[
    df["rule2_removed_chars"] > 0
].head(20)

for index, row in modified.iterrows():

    print(f"\n--- Ticket row {index} ---")

    print("\nAFTER RULE 1:")
    print(row["description_rule1"])

    print("\nAFTER RULE 2:")
    print(row["description_rule2"])

RULE 2 — PREPROCESSING VALIDATION

Original tickets: 8469
Empty after Rule 2: 0
Tickets modified by Rule 2: 124
Total characters removed: 2036
Average characters removed per ticket: 0.2404

RULE 2 BEFORE / AFTER

--- Ticket row 0 ---

AFTER RULE 1:
I'm having an issue with the GoPro Hero. Your billing zip code is: 71701. We appreciate that you have requested a website address. Please double check your email address. I've tried troubleshooting steps mentioned in the user manual, but the issue persists.

AFTER RULE 2:
I'm having an issue with the GoPro Hero. Please double check your email address. I've tried troubleshooting steps mentioned in the user manual, but the issue persists.

--- Ticket row 3 ---

AFTER RULE 1:
I'm having an issue with the Microsoft Office. If you have a problem you're interested in and I'd love to see this happen, please check out the Feedback. I've already contacted customer support multiple times, but the issue remains unresolved.

AFTER RULE 2:
I'm having an 

In [12]:
# ============================================================
# PREPROCESSING RULE 2 — REVISED
# Remove clearly unrelated injected content
# ============================================================
#
# Input:
#     description_rule1
#
# Output:
#     description_rule2
#
# Rule 2 is intentionally conservative.
# It removes known contamination fragments while preserving
# genuine support-ticket information.
# ============================================================

import re


def clean_injected_content(text: str) -> str:
    value = str(text)

    # --------------------------------------------------------
    # 1. Remove explicit source-code / game-log contamination
    # --------------------------------------------------------

    value = re.sub(
        r"\[[0-9]{1,2}:\d{2}:\d{2}(?:\.\d+)?\]"
        r"\s*\[[^\]]+\]"
        r"(?:\s*\[[^\]]+\])?"
        r"\s*"
        r".*?(?="
        r"\b(?:I['’]?m|I['’]?ve|I am|"
        r"I've|I've noticed|I'm experiencing|"
        r"I've checked|I've tried|"
        r"This problem|The issue|"
        r"I suspect|I need|I rely)\b"
        r"|$)",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"/\*.*?\*/",
        "",
        value,
        flags=re.DOTALL,
    )

    value = re.sub(
        r"\b(?:errno\s*=\s*0|"
        r"net\.minecraft\.inventory\.ItemStack)\b",
        "",
        value,
        flags=re.IGNORECASE,
    )

    # --------------------------------------------------------
    # 2. Remove explicit malformed injected fragments
    # --------------------------------------------------------

    value = re.sub(
        r"\bProduct Search:\s*What's New in 2-3-4-5\?",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bReport Feedback Customer Service is your best\b",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bplease send a ['\"]request['\"] to\s+\w+",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bRetain Seller Info\b",
        "",
        value,
        flags=re.IGNORECASE,
    )

    # --------------------------------------------------------
    # 3. Remove explicit marketplace contamination
    # --------------------------------------------------------

    value = re.sub(
        r"\bThe seller has sold the item for \$[\d,.]+\.",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bThe seller is not responsible for any damages "
        r"arising out of the delivery of the battleground game\.",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bPlease have the game in good condition and shipped to you\b",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bFirst, look at the seller['’]?s web site\.",
        "",
        value,
        flags=re.IGNORECASE,
    )

    # --------------------------------------------------------
    # 4. Remove explicit supplier / inventory contamination
    # --------------------------------------------------------

    value = re.sub(
        r"\(?Thanks\)? I will contact all my suppliers and confirm\.",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bPlease try and find out whether their inventory is "
        r"currently stocked, or any other reason\.",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bWhen we are the only supplier to provide this service\b",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bIf we can't keep this up, and the supplier can't figure "
        r"this out, we should have sent something in this particular "
        r"order right away\.",
        "",
        value,
        flags=re.IGNORECASE,
    )

    # --------------------------------------------------------
    # 5. Remove explicit feedback / website contamination
    # --------------------------------------------------------

    value = re.sub(
        r"\bWe appreciate that you have requested a website address\.",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bIf you have a problem you're interested in and I'd love "
        r"to see this happen, please check out the Feedback\.",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bIf you have any further feedback, please drop me an "
        r"email at [A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\.",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bPlease send feedback here -V\.3\.0\b",
        "",
        value,
        flags=re.IGNORECASE,
    )

    # --------------------------------------------------------
    # 6. Remove explicit transaction fragments
    # --------------------------------------------------------

    value = re.sub(
        r"\bYour billing zip code:\s*\d+\.",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bYour billing zip code is:\s*\d+\.",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bThe seller says on \d{1,2}/\d{1,2}/\d{2,4}\b",
        "",
        value,
        flags=re.IGNORECASE,
    )

    # --------------------------------------------------------
    # 7. Remove obvious metadata/injection labels
    # --------------------------------------------------------

    value = re.sub(
        r"\bCustomer Reviewer:\s*",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bUser Info:\s*"
        r"[A-Za-z0-9_.-]+\s+"
        r"[A-Za-z0-9_.-]+\s+"
        r"\d+\s+years?\s+ago\b",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\[[A-Z][A-Z0-9_ -]+\]\s*",
        "",
        value,
    )

    # --------------------------------------------------------
    # 8. Remove clearly orphaned contamination prefixes
    # --------------------------------------------------------

    value = re.sub(
        r"\bIf we can,\s*(?=The issue|This problem|"
        r"I['’]?m|I['’]?ve|I am)\b",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bI am\s+(?=I['’]?ve|I've|I'm|"
        r"I've performed|I['’]?m experiencing)\b",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bNote:\s*(?=I've|I['’]?ve|I'm|I['’]?m|"
        r"This problem|The issue)\b",
        "",
        value,
        flags=re.IGNORECASE,
    )

    # --------------------------------------------------------
    # 9. Remove standalone post/comment metadata
    # --------------------------------------------------------

    value = re.sub(
        r"\[This post has been updated\.\]",
        "",
        value,
        flags=re.IGNORECASE,
    )

    # --------------------------------------------------------
    # 10. Whitespace normalization
    # --------------------------------------------------------

    value = re.sub(r"\s{2,}", " ", value)
    value = re.sub(r"\s+([,.!?])", r"\1", value)

    return value.strip()


# ------------------------------------------------------------
# Apply Rule 2
# ------------------------------------------------------------

df["description_rule2"] = (
    df["description_rule1"]
    .astype(str)
    .apply(clean_injected_content)
)


# ------------------------------------------------------------
# Validation statistics
# ------------------------------------------------------------

df["rule1_char_count"] = df["description_rule1"].str.len()

df["rule2_char_count"] = (
    df["description_rule2"].str.len()
)

df["rule2_removed_chars"] = (
    df["rule1_char_count"]
    - df["rule2_char_count"]
)

df["rule2_empty"] = (
    df["description_rule2"].str.strip() == ""
)


print("=" * 70)
print("RULE 2 — REVISED VALIDATION")
print("=" * 70)

print(
    "\nOriginal tickets:",
    len(df)
)

print(
    "Empty after Rule 2:",
    int(df["rule2_empty"].sum())
)

print(
    "Tickets modified by Rule 2:",
    int((df["rule2_removed_chars"] > 0).sum())
)

print(
    "Total characters removed:",
    int(df["rule2_removed_chars"].sum())
)

print(
    "Average characters removed per ticket:",
    round(
        df["rule2_removed_chars"].mean(),
        4
    )
)


# ------------------------------------------------------------
# Before / after examples
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("RULE 2 BEFORE / AFTER")
print("=" * 70)

modified = df[
    df["rule2_removed_chars"] > 0
].head(20)

for index, row in modified.iterrows():

    print(f"\n--- Ticket row {index} ---")

    print("\nAFTER RULE 1:")
    print(row["description_rule1"])

    print("\nAFTER RULE 2:")
    print(row["description_rule2"])

RULE 2 — REVISED VALIDATION

Original tickets: 8469
Empty after Rule 2: 0
Tickets modified by Rule 2: 1071
Total characters removed: 15837
Average characters removed per ticket: 1.87

RULE 2 BEFORE / AFTER

--- Ticket row 0 ---

AFTER RULE 1:
I'm having an issue with the GoPro Hero. Your billing zip code is: 71701. We appreciate that you have requested a website address. Please double check your email address. I've tried troubleshooting steps mentioned in the user manual, but the issue persists.

AFTER RULE 2:
I'm having an issue with the GoPro Hero. Please double check your email address. I've tried troubleshooting steps mentioned in the user manual, but the issue persists.

--- Ticket row 3 ---

AFTER RULE 1:
I'm having an issue with the Microsoft Office. If you have a problem you're interested in and I'd love to see this happen, please check out the Feedback. I've already contacted customer support multiple times, but the issue remains unresolved.

AFTER RULE 2:
I'm having an issue 

In [13]:
# ============================================================
# PREPROCESSING RULE 2 — FINAL PATCH
# Remove clearly unrelated injected content
# while preserving genuine support information
# ============================================================

import re


def clean_injected_content(text: str) -> str:

    value = str(text)

    # --------------------------------------------------------
    # 1. SOURCE CODE / GAME LOGS
    # --------------------------------------------------------

    value = re.sub(
        r"\[[0-9]{1,2}:\d{2}:\d{2}(?:\.\d+)?\]"
        r"\s*\[[^\]]+\]"
        r"(?:\s*\[[^\]]+\])?"
        r"\s*"
        r".*?(?="
        r"\b(?:I['’]?m|I['’]?ve|I am|"
        r"I['’]?ve noticed|"
        r"I['’]?m experiencing|"
        r"I['’]?ve checked|"
        r"I['’]?ve tried|"
        r"This problem|The issue|"
        r"I suspect|I need|I rely)\b"
        r"|$)",
        "",
        value,
        flags=re.IGNORECASE,
    )

    # C-style comments
    value = re.sub(
        r"/\*.*?\*/",
        "",
        value,
        flags=re.DOTALL,
    )

    # Common code fragments
    value = re.sub(
        r"\b(?:errno\s*=\s*0|net\.minecraft\.inventory\.ItemStack)\b",
        "",
        value,
        flags=re.IGNORECASE,
    )

    # Java/code remnants
    value = re.sub(
        r"\b(?:@param|@return)\b",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bif\s*\([^)]*_[^)]*\)",
        "",
        value,
        flags=re.IGNORECASE,
    )


    # --------------------------------------------------------
    # 2. MALFORMED / NON-SUPPORT CONTENT
    # --------------------------------------------------------

    value = re.sub(
        r"\bProduct Search:\s*What's New in 2-3-4-5\?",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bReport Feedback Customer Service is your best\b",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bplease send a ['\"]request['\"] to\s+\w+",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bRetain Seller Info\b",
        "",
        value,
        flags=re.IGNORECASE,
    )


    # --------------------------------------------------------
    # 3. MARKETPLACE / SELLER INJECTION
    # --------------------------------------------------------

    value = re.sub(
        r"\bThe seller has sold the item for \$[\d,.]+\.",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bThe seller is not responsible for any damages arising out of the delivery of the battleground game\.",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bPlease have the game in good condition and shipped to you\b",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bFirst, look at the seller['’]?s web site\.",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bWhen we receive your replacement, we will exchange the item for a full refund or exchange immediately upon\b",
        "",
        value,
        flags=re.IGNORECASE,
    )


    # --------------------------------------------------------
    # 4. SUPPLIER / INVENTORY
    # --------------------------------------------------------

    value = re.sub(
        r"\(?Thanks\)? I will contact all my suppliers and confirm\.",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bPlease try and find out whether their inventory is currently stocked, or any other reason\.",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bWhen we are the only supplier to provide this service\b",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bIf we can't keep this up, and the supplier can't figure this out, we should have sent something in this particular order right away\.",
        "",
        value,
        flags=re.IGNORECASE,
    )


    # --------------------------------------------------------
    # 5. FEEDBACK / WEBSITE INJECTION
    # --------------------------------------------------------

    value = re.sub(
        r"\bWe appreciate that you have requested a website address\.",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bIf you have a problem you're interested in and I'd love to see this happen, please check out the Feedback\.",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bIf you have any further feedback, please drop me an email at [A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\.",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bPlease send feedback here -V\.3\.0\b",
        "",
        value,
        flags=re.IGNORECASE,
    )


    # --------------------------------------------------------
    # 6. TRANSACTION / BILLING INJECTION
    # --------------------------------------------------------

    value = re.sub(
        r"\bYour billing zip code:\s*\d+\.",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bYour billing zip code is:\s*\d+\.",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bThe seller says on \d{1,2}/\d{1,2}/\d{2,4}\b",
        "",
        value,
        flags=re.IGNORECASE,
    )


    # --------------------------------------------------------
    # 7. METADATA INJECTION
    # --------------------------------------------------------

    # Remove label + obvious reviewer fragment
    value = re.sub(
        r"\bCustomer Reviewer:\s*"
        r"(?:My Husband|My Wife|My Friend|My Brother|My Sister|"
        r"My Father|My Mother|My Son|My Daughter)\b"
        r".*?(?=\bI['’]?ve|\bI['’]?m|\bThis problem|\bThe issue)",
        "",
        value,
        flags=re.IGNORECASE,
    )

    # User metadata
    value = re.sub(
        r"\bUser Info:\s*"
        r"[A-Za-z0-9_.-]+\s+"
        r"[A-Za-z0-9_.-]+\s+"
        r"\d+\s+years?\s+ago\b",
        "",
        value,
        flags=re.IGNORECASE,
    )

    # Post metadata
    value = re.sub(
        r"\[This post has been updated\.\]",
        "",
        value,
        flags=re.IGNORECASE,
    )


    # --------------------------------------------------------
    # 8. MALFORMED NUMBERED FRAGMENTS
    # --------------------------------------------------------

    # Example: "1- [PHONE] . Product Search..."
    value = re.sub(
        r"\b\d+\s*-\s*(?:\[[A-Z_]+\])?\s*\.?",
        "",
        value,
        flags=re.IGNORECASE,
    )

    # Example: "#4: The product. #5: GoPro Action Camera."
    value = re.sub(
        r"#\d+\s*:\s*[^#.!?]*(?:[.!?]|$)",
        "",
        value,
        flags=re.IGNORECASE,
    )


    # --------------------------------------------------------
    # 9. PLACEHOLDER HANDLING
    # --------------------------------------------------------

    # Remove complete sentence whose only useful content
    # is a placeholder.
    value = re.sub(
        r"\bIt says\s*['\"]?\s*\[PLACEHOLDER\]\s*['\"]?\s*\.",
        "",
        value,
        flags=re.IGNORECASE,
    )

    # Remove placeholder token while preserving sentence.
    value = re.sub(
        r"\[PLACEHOLDER\]",
        "",
        value,
        flags=re.IGNORECASE,
    )


    # --------------------------------------------------------
    # 10. GENERIC MALFORMED PRODUCT VARIABLE
    # --------------------------------------------------------

    value = re.sub(
        r"\bI'm having an issue with the\s+product_purchased\s*\.",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bI'm having an issue with the\s*\.",
        "",
        value,
        flags=re.IGNORECASE,
    )


    # --------------------------------------------------------
    # 11. REMAINING ORPHAN PREFIXES
    # --------------------------------------------------------

    value = re.sub(
        r"\bIf we can,\s*(?=The issue|This problem|I['’]?m|I['’]?ve|I am)",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bI am\s+(?=I['’]?ve|I['’]?m|I've performed|I['’]?m experiencing)",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bNote:\s*(?=I've|I['’]?ve|I'm|I['’]?m|This problem|The issue)",
        "",
        value,
        flags=re.IGNORECASE,
    )


    # --------------------------------------------------------
    # 12. URL / PHONE PLACEHOLDER CLEANUP
    # --------------------------------------------------------

    # Remove explicit URL placeholder.
    value = re.sub(
        r"\[URL\]",
        "",
        value,
        flags=re.IGNORECASE,
    )

    # Remove phone placeholder only.
    # Do NOT remove surrounding support sentence.
    value = re.sub(
        r"\[PHONE\]",
        "",
        value,
        flags=re.IGNORECASE,
    )


    # --------------------------------------------------------
    # 13. WHITESPACE / PUNCTUATION
    # --------------------------------------------------------

    value = re.sub(
        r"\s{2,}",
        " ",
        value,
    )

    value = re.sub(
        r"\s+([,.!?])",
        r"\1",
        value,
    )

    value = re.sub(
        r"\.{2,}",
        ".",
        value,
    )

    # Remove empty quote artifacts
    value = re.sub(
        r"['\"]{2,}",
        "",
        value,
    )

    # Remove repeated punctuation
    value = re.sub(
        r"\s+\.",
        ".",
        value,
    )

    return value.strip()

In [14]:
df["description_rule2_final"] = (
    df["description_rule1"]
    .astype(str)
    .apply(clean_injected_content)
)

df["rule2_final_empty"] = (
    df["description_rule2_final"].str.strip() == ""
)

df["rule2_final_char_count"] = (
    df["description_rule2_final"].str.len()
)

df["rule2_final_removed_chars"] = (
    df["description_rule1"].str.len()
    - df["rule2_final_char_count"]
)

print("=" * 70)
print("RULE 2 — FINAL PATCH VALIDATION")
print("=" * 70)

print("\nOriginal tickets:", len(df))

print(
    "Empty after Rule 2:",
    int(df["rule2_final_empty"].sum())
)

print(
    "Tickets modified by Rule 2:",
    int((df["rule2_final_removed_chars"] > 0).sum())
)

print(
    "Total characters removed:",
    int(df["rule2_final_removed_chars"].sum())
)

print(
    "Average characters removed per ticket:",
    round(df["rule2_final_removed_chars"].mean(), 4)
)

RULE 2 — FINAL PATCH VALIDATION

Original tickets: 8469
Empty after Rule 2: 0
Tickets modified by Rule 2: 1368
Total characters removed: 25305
Average characters removed per ticket: 2.988


In [15]:
check_rows = [
    10, 19, 31, 49, 51,
    64, 65, 70, 73,
    86, 93, 95, 99,
    108, 118, 131, 153,
    164, 221, 478, 505,
    507, 516, 520
]

for index in check_rows:

    print("\n" + "=" * 70)
    print(f"TICKET ROW {index}")
    print("=" * 70)

    print("\nAFTER RULE 1:")
    print(df.loc[index, "description_rule1"])

    print("\nAFTER RULE 2:")
    print(df.loc[index, "description_rule2_final"])


TICKET ROW 10

AFTER RULE 1:
I'm having an issue with the Nintendo Switch. 1- [PHONE] . Product Search: What's New in 2-3-4-5? Report Feedback Customer Service is your best I'm using the original charger that came with my Nintendo Switch, but it's not charging properly.

AFTER RULE 2:
I'm having an issue with the Nintendo Switch. I'm using the original charger that came with my Nintendo Switch, but it's not charging properly.

TICKET ROW 19

AFTER RULE 1:
I'm having an issue with the Canon EOS. I'm having an issue with the product_purchased . Customer Reviewer: My Husband was able to take an order from Apple I've checked for any available software updates for my Canon EOS, but there are none.

AFTER RULE 2:
I'm having an issue with the Canon EOS. I've checked for any available software updates for my Canon EOS, but there are none.

TICKET ROW 31

AFTER RULE 1:
I'm having an issue with the GoPro Action Camera. The seller has sold the item for $29.89. When we receive your replacement, w

In [16]:
# ============================================================
# RULE 2 — FINAL TARGETED ARTIFACT CLEANUP
# ============================================================

# 1. Remove malformed refund/call fragment
value = re.sub(
    r"\bOtherwise,\s*please call us at\.",
    "",
    value,
    flags=re.IGNORECASE,
)

# 2. Remove orphan "I'm having an issue with the"
value = re.sub(
    r"\bI'm having an issue with the\s+(?=I'm concerned|This problem|The issue|$)",
    "",
    value,
    flags=re.IGNORECASE,
)

# 3. Remove orphan URL instruction
value = re.sub(
    r"\b\d+\.\s*Visit\s*(?=I've|I'm|The|This|I\b)",
    "",
    value,
    flags=re.IGNORECASE,
)

# 4. Remove remaining obvious code annotation residue
value = re.sub(
    r"\*+\s*@param\s+_?\w+\s*\*+\s*@return\s*@\*/\)?",
    "",
    value,
    flags=re.IGNORECASE,
)

# 5. Remove obvious blockchain injection
value = re.sub(
    r'["\']\s*,\s*["\']\s*;\s*Here,\s*we\s+can\s+fetch\s+a\s+blockchain\b',
    "",
    value,
    flags=re.IGNORECASE,
)

# 6. Remove boilerplate attached to previous sentence
value = re.sub(
    r"(?i)\bPavilion\.Please assist\.",
    "Pavilion.",
    value,
)

# 7. Remove orphan quotation artifact
value = re.sub(
    r'\.\s*"\s+(?=I\'ve|I\'m|This|The)',
    ". ",
    value,
)

# 8. Remove duplicated generic support instruction
value = re.sub(
    r"\bContact customer support\.\s*Contact customer support\.",
    "",
    value,
    flags=re.IGNORECASE,
)

# 9. Remove orphan generic fragment
value = re.sub(
    r"\bI'm having an issue with some\s+(?=I'm unable|I['’]m unable|The issue|This problem)",
    "",
    value,
    flags=re.IGNORECASE,
)

# 10. Remove broken fragment observed in row 478
value = re.sub(
    r"\bThe base plate and the\s+['\"]C['\"]\s+on\b",
    "",
    value,
    flags=re.IGNORECASE,
)

# 11. Final whitespace / punctuation cleanup
value = re.sub(r"\s{2,}", " ", value)
value = re.sub(r"\s+([,.!?])", r"\1", value)
value = re.sub(r"\.{2,}", ".", value)
value = value.strip()

In [17]:
print("=" * 70)
print("RULE 2 — FINAL ACCEPTANCE CHECK")
print("=" * 70)

print("Original tickets:", len(df))

print(
    "Empty after Rule 2:",
    int((df["description_rule2_final"].str.strip() == "").sum())
)

print(
    "Modified tickets:",
    int((df["description_rule2_final"] != df["description_rule1"]).sum())
)

print(
    "Characters removed:",
    int(
        (
            df["description_rule1"].str.len()
            - df["description_rule2_final"].str.len()
        ).sum()
    )
)

RULE 2 — FINAL ACCEPTANCE CHECK
Original tickets: 8469
Empty after Rule 2: 0
Modified tickets: 1368
Characters removed: 25305


In [18]:
final_check_rows = [
    49, 51, 86, 108, 118,
    131, 221, 478
]

for index in final_check_rows:

    print("\n" + "=" * 70)
    print(f"TICKET ROW {index}")
    print("=" * 70)

    print("\nAFTER RULE 1:")
    print(df.loc[index, "description_rule1"])

    print("\nFINAL RULE 2:")
    print(df.loc[index, "description_rule2_final"])


TICKET ROW 49

AFTER RULE 1:
I'm having an issue with the Microsoft Office. If you've received a refund, please contact Customer Service. Otherwise, please call us at [PHONE] . I'm having an issue with the I'm concerned about the security of my Microsoft Office and would like to ensure that my data is safe.

FINAL RULE 2:
I'm having an issue with the Microsoft Office. If you've received a refund, please contact Customer Service. Otherwise, please call us at. I'm having an issue with the I'm concerned about the security of my Microsoft Office and would like to ensure that my data is safe.

TICKET ROW 51

AFTER RULE 1:
I've recently set up my LG Smart TV, but it fails to connect to any available networks. What steps should I take to troubleshoot this issue?.......................................................1. Visit [URL] I've reviewed the troubleshooting steps on the official support website, but they didn't resolve the problem.

FINAL RULE 2:
I've recently set up my LG Smart TV, bu

In [19]:
# ============================================================
# FINAL TARGETED ARTIFACT CLEANUP
# ============================================================

# Row 49 — remove broken phone/call fragment
value = re.sub(
    r"Otherwise,\s*please call us at\.",
    "",
    value,
    flags=re.IGNORECASE,
)

# Row 49 — remove incomplete generic opening
value = re.sub(
    r"\bI'm having an issue with the\s+(?=I'm concerned\b)",
    "",
    value,
    flags=re.IGNORECASE,
)


# Row 51 — remove the entire leftover "1. Visit" fragment
value = re.sub(
    r"\.{2,}\s*\d+\.\s*Visit\s*",
    ". ",
    value,
    flags=re.IGNORECASE,
)


# Row 86 — remove remaining code annotation residue
value = re.sub(
    r"\*+\s*@param\s*_?\w+\s*\*+\s*@return\s*@\*/\)?",
    "",
    value,
    flags=re.IGNORECASE,
)


# Row 108 — remove obvious blockchain injection
value = re.sub(
    r'["\']\s*,\s*["\']\s*["\']?\s*;\s*Here,\s*we\s+can\s+fetch\s+a\s+blockchain\b',
    "",
    value,
    flags=re.IGNORECASE,
)


# Row 118 — separate attached "Please assist."
value = re.sub(
    r"(?i)(Pavilion)\.Please assist\.",
    r"\1.",
    value,
)


# Row 131 — remove orphan quotation left by metadata removal
value = re.sub(
    r"\.\s*\"\s*(?=I've recently updated\b)",
    ". ",
    value,
    flags=re.IGNORECASE,
)


# Row 221 — remove duplicated generic support phrase
value = re.sub(
    r"\bContact customer support\.\s*Contact customer support\.",
    "",
    value,
    flags=re.IGNORECASE,
)

# Remove malformed generic fragment before genuine issue
value = re.sub(
    r"\bI'm having an issue with some\s+(?=I'm unable to find\b)",
    "",
    value,
    flags=re.IGNORECASE,
)


# Row 478 — remove clearly truncated injected fragment
value = re.sub(
    r"\bThe base plate and the\s+['\"]C['\"]\s+on\b",
    "",
    value,
    flags=re.IGNORECASE,
)


# Final normalization
value = re.sub(r"\s{2,}", " ", value)
value = re.sub(r"\s+([,.!?])", r"\1", value)
value = re.sub(r"\.{2,}", ".", value)
value = value.strip()

In [20]:
df["description_rule2_final"] = (
    df["description_rule1"]
    .astype(str)
    .apply(clean_injected_content)
)

In [21]:
final_check_rows = [
    49, 51, 86, 108,
    118, 131, 221, 478
]

for index in final_check_rows:

    print("\n" + "=" * 70)
    print(f"TICKET ROW {index}")
    print("=" * 70)

    print("\nFINAL RULE 2:")
    print(df.loc[index, "description_rule2_final"])


TICKET ROW 49

FINAL RULE 2:
I'm having an issue with the Microsoft Office. If you've received a refund, please contact Customer Service. Otherwise, please call us at. I'm having an issue with the I'm concerned about the security of my Microsoft Office and would like to ensure that my data is safe.

TICKET ROW 51

FINAL RULE 2:
I've recently set up my LG Smart TV, but it fails to connect to any available networks. What steps should I take to troubleshoot this issue?.1. Visit I've reviewed the troubleshooting steps on the official support website, but they didn't resolve the problem.

TICKET ROW 86

FINAL RULE 2:
I'm having an issue with the Sony PlayStation. * * * @param _item * * @return @*/) I've noticed that the issue occurs consistently when I use a specific feature or application on my Sony PlayStation.

TICKET ROW 108

FINAL RULE 2:
I'm having an issue with the Dyson Vacuum Cleaner. ", " " ; Here, we can fetch a blockchain I've noticed a sudden decrease in battery life on my Dys

In [23]:
import re


def clean_injected_content(text: str) -> str:

    value = str(text)

    # ========================================================
    # 1. Clearly unrelated marketplace / seller content
    # ========================================================

    patterns = [

        r"The seller has sold the item for \$[\d,.]+\.",
        
        r"The seller is not responsible for any damages arising out of the delivery of the battleground game\.",

        r"Please have the game in good condition and shipped to you",

        r"First, look at the seller['’]?s web site\.",

        r"When we receive your replacement, we will exchange the item for a full refund or exchange immediately upon",

        r"The seller says on \d{1,2}/\d{1,2}/\d{2,4}",

    ]

    for pattern in patterns:
        value = re.sub(
            pattern,
            "",
            value,
            flags=re.IGNORECASE,
        )


    # ========================================================
    # 2. Supplier / inventory contamination
    # ========================================================

    patterns = [

        r"\(?Thanks\)? I will contact all my suppliers and confirm\.",

        r"Please try and find out whether their inventory is currently stocked, or any other reason\.",

        r"When we are the only supplier to provide this service",

        r"If we can't keep this up, and the supplier can't figure this out, we should have sent something in this particular order right away\.",

    ]

    for pattern in patterns:
        value = re.sub(
            pattern,
            "",
            value,
            flags=re.IGNORECASE,
        )


    # ========================================================
    # 3. Feedback / website contamination
    # ========================================================

    patterns = [

        r"We appreciate that you have requested a website address\.",

        r"If you have a problem you're interested in and I'd love to see this happen, please check out the Feedback\.",

        r"If you have any further feedback, please drop me an email at [A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\.",

        r"Please send feedback here -V\.3\.0",

    ]

    for pattern in patterns:
        value = re.sub(
            pattern,
            "",
            value,
            flags=re.IGNORECASE,
        )


    # ========================================================
    # 4. Billing / transaction contamination
    # ========================================================

    value = re.sub(
        r"Your billing zip code(?: is)?:\s*\d+\.",
        "",
        value,
        flags=re.IGNORECASE,
    )


    # ========================================================
    # 5. Obvious metadata contamination
    # ========================================================

    value = re.sub(
        r"Customer Reviewer:\s*"
        r"(?:My Husband|My Wife|My Friend|My Brother|My Sister|"
        r"My Father|My Mother|My Son|My Daughter)"
        r".*?(?=I've|I'm|This problem|The issue)",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"User Info:\s*"
        r"[A-Za-z0-9_.-]+\s+"
        r"[A-Za-z0-9_.-]+\s+"
        r"\d+\s+years?\s+ago",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\[This post has been updated\.\]",
        "",
        value,
        flags=re.IGNORECASE,
    )


    # ========================================================
    # 6. Source-code contamination
    # ========================================================

    # Remove complete C-style comments
    value = re.sub(
        r"/\*.*?\*/",
        "",
        value,
        flags=re.DOTALL,
    )

    # Remove Java annotation remnants
    value = re.sub(
        r"\*+\s*@param\s+[A-Za-z0-9_]+\s*\*+\s*@return\s*",
        "",
        value,
        flags=re.IGNORECASE,
    )

    # Remove function/code tail observed in corpus
    value = re.sub(
        r"@?\*/?\s*if\s*\(_[A-Za-z0-9_]+\([^)]*\)\)\s*",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\b(?:errno\s*=\s*0|net\.minecraft\.inventory\.ItemStack)\b",
        "",
        value,
        flags=re.IGNORECASE,
    )


    # ========================================================
    # 7. Placeholder-only error messages
    # ========================================================

    value = re.sub(
        r"\bI've noticed a peculiar error message popping up on "
        r"[^.]+\. It says\s*['\"]?\s*\[PLACEHOLDER\]\s*['\"]?\.",
        "",
        value,
        flags=re.IGNORECASE,
    )


    # ========================================================
    # 8. Remaining placeholders
    # ========================================================

    value = re.sub(
        r"\[PLACEHOLDER\]",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\[URL\]",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\[PHONE\]",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\[CODE_REMOVED\]",
        "",
        value,
        flags=re.IGNORECASE,
    )


    # ========================================================
    # 9. Malformed product variable
    # ========================================================

    value = re.sub(
        r"\bI'm having an issue with the\s+product_purchased\s*\.",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bI'm having an issue with the\s*\.",
        "",
        value,
        flags=re.IGNORECASE,
    )


    # ========================================================
    # 10. Malformed numbered fragments
    # ========================================================

    value = re.sub(
        r"\b\d+\s*-\s*(?:\[[A-Z_]+\])?\s*\.?",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"#\d+\s*:\s*[^#.!?]*(?:[.!?]|$)",
        "",
        value,
        flags=re.IGNORECASE,
    )


    # ========================================================
    # 11. Known blockchain injection
    # ========================================================

    value = re.sub(
        r'["\']\s*,\s*["\']\s*;\s*Here,\s*we\s+can\s+fetch\s+a\s+blockchain\b',
        "",
        value,
        flags=re.IGNORECASE,
    )


    # ========================================================
    # 12. Known orphan fragments
    # ========================================================

    value = re.sub(
        r"\bIf we can,\s*(?=The issue|This problem|I'm|I've)",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bI am\s+(?=I've|I'm)",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bNote:\s*(?=I've|I'm|This problem|The issue)",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bI'm having an issue with some\s+(?=I'm unable to find)",
        "",
        value,
        flags=re.IGNORECASE,
    )


    # ========================================================
    # 13. Attached generic boilerplate
    # ========================================================

    value = re.sub(
        r"(?i)(Pavilion)\.Please assist\.",
        r"\1.",
        value,
    )

    value = re.sub(
        r"\bContact customer support\.\s*Contact customer support\.",
        "",
        value,
        flags=re.IGNORECASE,
    )

    value = re.sub(
        r"\bOtherwise,\s*please call us at\.",
        "",
        value,
        flags=re.IGNORECASE,
    )


    # ========================================================
    # 14. Broken URL instruction
    # ========================================================

    value = re.sub(
        r"\b\d+\.\s*Visit\s*(?=I've|I'm|The|This|I\b)",
        "",
        value,
        flags=re.IGNORECASE,
    )


    # ========================================================
    # 15. Known broken fragment from row 478
    # ========================================================

    value = re.sub(
        r"\bThe base plate and the\s+['\"]C['\"]\s+on\b",
        "",
        value,
        flags=re.IGNORECASE,
    )


    # ========================================================
    # 16. Orphan quotation artifacts
    # ========================================================

    value = re.sub(
        r"\.\s*\"\s+(?=I've|I'm|This|The)",
        ". ",
        value,
        flags=re.IGNORECASE,
    )


    # ========================================================
    # 17. Whitespace / punctuation normalization
    # ========================================================

    value = re.sub(
        r"\s+",
        " ",
        value,
    )

    value = re.sub(
        r"\s+([,.!?])",
        r"\1",
        value,
    )

    value = re.sub(
        r"\.{2,}",
        ".",
        value,
    )

    value = value.strip()

    return value

In [24]:
df["description_rule2_final"] = (
    df["description_rule1"]
    .astype(str)
    .apply(clean_injected_content)
)

In [25]:
final_check_rows = [
    49, 51, 86, 108,
    118, 131, 221, 478
]

for index in final_check_rows:

    print("\n" + "=" * 70)
    print(f"TICKET ROW {index}")
    print("=" * 70)

    print(df.loc[index, "description_rule2_final"])


TICKET ROW 49
I'm having an issue with the Microsoft Office. If you've received a refund, please contact Customer Service. Otherwise, please call us at. I'm having an issue with the I'm concerned about the security of my Microsoft Office and would like to ensure that my data is safe.

TICKET ROW 51
I've recently set up my LG Smart TV, but it fails to connect to any available networks. What steps should I take to troubleshoot this issue?.I've reviewed the troubleshooting steps on the official support website, but they didn't resolve the problem.

TICKET ROW 86
I'm having an issue with the Sony PlayStation. * * * @param _item * * @return I've noticed that the issue occurs consistently when I use a specific feature or application on my Sony PlayStation.

TICKET ROW 108
I'm having an issue with the Dyson Vacuum Cleaner. ", " " ; Here, we can fetch a blockchain I've noticed a sudden decrease in battery life on my Dyson Vacuum Cleaner. It used to last much longer.

TICKET ROW 118
I'm having

Ticket Description
        ↓
Rule 1
generic boilerplate removal
        ↓
Rule 2
confirmed contamination removal
        ↓
description_rule2_final
        ↓
BERTopic-targeted preprocessing
        ↓
BERTopic

In [26]:
# Final targeted artifact check before saving the BERTopic dataset

artifact_patterns = [
    "@param",
    "@return",
    "blockchain",
    "Otherwise, please call us at.",
    "I'm having an issue with some",
    'The base plate and the "C" on',
    'Here, we can fetch',
]

for pattern in artifact_patterns:
    matches = df[
        df["description_rule2"]
        .astype(str)
        .str.contains(pattern, case=False, regex=False, na=False)
    ]

    if len(matches):
        print(f"\n--- {pattern!r}: {len(matches)} matches ---")
        print(
            matches[["Ticket ID", "description_rule2"]]
            .to_string(index=False)
        )


--- '@param': 6 matches ---
 Ticket ID                                                                                                                                                                                                                                                                 description_rule2
        87                                           I'm having an issue with the Sony PlayStation. * * * @param _item * * @return @*/if (_ejection_exceeds(_proposal)) I've noticed that the issue occurs consistently when I use a specific feature or application on my Sony PlayStation.
      2876                                              I'm having an issue with the Nintendo Switch. * * @hide * * @param id integer id which can be the ID of all products we will have. This includes I'm unable to find the option to perform the desired action in the Nintendo Switch.
      4383                  I'm having an issue with the Nintendo Switch Pro Controller. * ** * @param psql As MySQL

In [30]:
import re


def final_rule2_cleanup(text: str) -> str:
    if not text:
        return ""

    text = str(text)

    # ---------------------------------------------------------
    # 1. Remove confirmed code / documentation contamination
    # ---------------------------------------------------------
    code_patterns = [
        r"\*+\s*@param\b.*?(?=I've|I'm|The issue|The problem|I’m|$)",
        r"\*+\s*@return\b.*?(?=I've|I'm|The issue|The problem|I’m|$)",
        r"@hide\b.*?(?=I'm unable|I've|I'm|The issue|$)",
        r"@\*/if\s*\([^)]*\).*?(?=I've|I'm|The issue|The problem|$)",
        r"/\*\*.*?(?=I've|I'm|The issue|The problem|$)",
        r"\bfunction\s+\w+\b.*?(?=I'm|I've|The issue|The problem|$)",
        r"\bget_products\b.*?(?=I'm|I've|The issue|The problem|$)",
        r"\bset_products\b.*?(?=I'm|I've|The issue|The problem|$)",
        r"\bpublic\s+void\s+\w+\s*\(.*?(?=I'm|I've|The issue|The problem|$)",
    ]

    for pattern in code_patterns:
        text = re.sub(pattern, " ", text, flags=re.IGNORECASE | re.DOTALL)

    # ---------------------------------------------------------
    # 2. Remove confirmed blockchain / generated-text fragments
    # ---------------------------------------------------------
    text = re.sub(
        r'"\s*,\s*"\s*"\s*;\s*Here,\s*we\s*can\s*fetch\s+a\s+blockchain',
        " ",
        text,
        flags=re.IGNORECASE,
    )

    # Remove the isolated blockchain injection in Ticket 4409
    text = re.sub(
        r"\bI\s+blockchain\s+based\s+on\s+my\s+experience\s+and\s+advice\b",
        " ",
        text,
        flags=re.IGNORECASE,
    )

    # ---------------------------------------------------------
    # 3. Remove confirmed broken / orphan fragments
    # ---------------------------------------------------------
    text = re.sub(
        r"\bOtherwise,\s*please\s*call\s*us\s*at\.",
        " ",
        text,
        flags=re.IGNORECASE,
    )

    text = re.sub(
        r"\bI'm\s+having\s+an\s+issue\s+with\s+the\s+I'm\s+concerned\b",
        "I'm concerned",
        text,
        flags=re.IGNORECASE,
    )

    text = re.sub(
        r"\bI'm\s+having\s+an\s+issue\s+with\s+the\s+product_purchased\.",
        " ",
        text,
        flags=re.IGNORECASE,
    )

    text = re.sub(
        r"\bI'm\s+having\s+an\s+issue\s+with\s+some\s+I'm\s+unable\b",
        "I'm unable",
        text,
        flags=re.IGNORECASE,
    )

    text = re.sub(
        r"\bThe\s+base\s+plate\s+and\s+the\s+\"C\"\s+on\b",
        " ",
        text,
        flags=re.IGNORECASE,
    )

    # ---------------------------------------------------------
    # 4. Remove orphan quotation marks
    # ---------------------------------------------------------
    text = re.sub(
        r"(?<=\.)\s*\"\s*(?=I've|I'm|The|This|Could|I)",
        " ",
        text,
        flags=re.IGNORECASE,
    )

    # ---------------------------------------------------------
    # 5. Remove duplicated generic support sentence
    # ---------------------------------------------------------
    text = re.sub(
        r"(Contact\s+customer\s+support\.)"
        r"(\s*\1)+",
        r"\1",
        text,
        flags=re.IGNORECASE,
    )

    # ---------------------------------------------------------
    # 6. Clean whitespace / punctuation artifacts
    # ---------------------------------------------------------
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+([,.!?])", r"\1", text)
    text = re.sub(r"([.!?])(?=[A-Z])", r"\1 ", text)

    return text.strip()

In [31]:
df["bertopic_text"] = (
    df["description_rule2"]
    .astype(str)
    .apply(final_rule2_cleanup)
)

In [32]:
remaining_patterns = [
    "@param",
    "@return",
    "@hide",
    "blockchain",
    "Here, we can fetch",
    "Otherwise, please call us at.",
    "I'm having an issue with some",
    'The base plate and the "C" on',
    "product_purchased.",
]

for pattern in remaining_patterns:
    matches = df[
        df["bertopic_text"]
        .str.contains(pattern, case=False, regex=False, na=False)
    ]

    print(f"{pattern!r}: {len(matches)} matches")

'@param': 0 matches
'@return': 0 matches
'@hide': 0 matches
'blockchain': 0 matches
'Here, we can fetch': 0 matches
'Otherwise, please call us at.': 0 matches
"I'm having an issue with some": 0 matches
'The base plate and the "C" on': 0 matches
'product_purchased.': 9 matches


In [33]:
check_ids = [50, 87, 109, 222, 2876, 4383, 4393, 4409, 479, 6518, 6805]

df.loc[
    df["Ticket ID"].isin(check_ids),
    ["Ticket ID", "Ticket Subject", "bertopic_text"]
].to_string(index=False)

" Ticket ID           Ticket Subject                                                                                                                                                                                                                                                                         bertopic_text\n        50            Display issue                                                                   I'm having an issue with the Microsoft Office. If you've received a refund, please contact Customer Service. I'm concerned about the security of my Microsoft Office and would like to ensure that my data is safe.\n        87   Product recommendation                                                                                                               I'm having an issue with the Sony PlayStation. * * I've noticed that the issue occurs consistently when I use a specific feature or application on my Sony PlayStation.\n       109            Product setup                   

In [34]:
df["bertopic_text"] = (
    df["bertopic_text"]
    .str.replace(r"\*/", " ", regex=True)
    .str.replace(r"\* \* \*", " ", regex=True)
    .str.replace(r"\* \*\*", " ", regex=True)
    .str.replace(r"\* \*", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [35]:
check_ids = [50, 87, 109, 222, 479, 2876, 4383, 4393, 4409, 6518, 6805]

df.loc[
    df["Ticket ID"].isin(check_ids),
    ["Ticket ID", "Ticket Subject", "bertopic_text"]
].to_string(index=False)

" Ticket ID           Ticket Subject                                                                                                                                                                                                                                                                         bertopic_text\n        50            Display issue                                                                   I'm having an issue with the Microsoft Office. If you've received a refund, please contact Customer Service. I'm concerned about the security of my Microsoft Office and would like to ensure that my data is safe.\n        87   Product recommendation                                                                                                                   I'm having an issue with the Sony PlayStation. I've noticed that the issue occurs consistently when I use a specific feature or application on my Sony PlayStation.\n       109            Product setup                   

In [36]:
from pathlib import Path

# Project-level processed-data directory
output_dir = Path("../../../../../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

# Keep only the information needed for BERTopic modelling
bertopic_dataset = df[
    [
        "Ticket ID",
        "Ticket Subject",
        "Ticket Description",
        "bertopic_text",
    ]
].copy()

output_path = output_dir / "supportmind_bertopic_cleaned_dataset.csv"

bertopic_dataset.to_csv(output_path, index=False)

print("BERTopic dataset saved successfully.")
print(f"Path: {output_path.resolve()}")
print(f"Rows: {len(bertopic_dataset):,}")
print(f"Columns: {list(bertopic_dataset.columns)}")

BERTopic dataset saved successfully.
Path: C:\Users\danie\supportmind-ai\data\processed\supportmind_bertopic_cleaned_dataset.csv
Rows: 8,469
Columns: ['Ticket ID', 'Ticket Subject', 'Ticket Description', 'bertopic_text']


In [37]:
import pandas as pd

text = bertopic_dataset["bertopic_text"].astype(str)

print("Documents:", len(text))
print("Empty:", (text.str.strip() == "").sum())
print("Characters:")
print(text.str.len().describe())
print("\nWords:")
print(text.str.split().str.len().describe())

Documents: 8469
Empty: 0
Characters:
count    8469.000000
mean      252.670799
std        44.935360
min        34.000000
25%       233.000000
50%       263.000000
75%       282.000000
max       364.000000
Name: bertopic_text, dtype: float64

Words:
count    8469.000000
mean       44.421892
std         8.529490
min         7.000000
25%        40.000000
50%        46.000000
75%        51.000000
max        65.000000
Name: bertopic_text, dtype: float64


In [38]:
# BERTopic-targeted residual noise check

patterns = {
    "placeholder": r"\[PLACEHOLDER\]",
    "url": r"https?://|www\.",
    "email": r"\b[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}\b",
    "code_marker": r"@param|@return|/\*|\*/|function\s+\w+",
}

for name, pattern in patterns.items():
    count = text.str.contains(
        pattern,
        case=False,
        regex=True,
        na=False
    ).sum()

    print(f"{name}: {count:,}")

placeholder: 0
url: 7
email: 72
code_marker: 0


In [39]:
# BERTopic-specific check:
# How much of the corpus contains product information?

product_names = (
    bertopic_dataset["Ticket Description"]
    .astype(str)
    .str.lower()
)

products = (
    df["Product Purchased"]
    .dropna()
    .astype(str)
    .str.lower()
    .unique()
)

product_hits = product_names.apply(
    lambda text: sum(product in text for product in products)
)

print("Documents:", len(product_hits))
print("Documents containing at least one product name:",
      (product_hits > 0).sum())
print("Percentage:",
      round((product_hits > 0).mean() * 100, 2), "%")

Documents: 8469
Documents containing at least one product name: 8469
Percentage: 100.0 %


In [40]:
import re

def prepare_bertopic_text(text: str) -> str:
    text = str(text)

    # Remove URLs
    text = re.sub(
        r"https?://\S+|www\.\S+",
        " ",
        text,
        flags=re.IGNORECASE
    )

    # Remove email addresses
    text = re.sub(
        r"\b[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}\b",
        " ",
        text
    )

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


bertopic_dataset["bertopic_input"] = (
    bertopic_dataset["bertopic_text"]
    .apply(prepare_bertopic_text)
)

print("Documents:", len(bertopic_dataset))
print(
    "Empty documents:",
    (bertopic_dataset["bertopic_input"].str.strip() == "").sum()
)
print(
    "Mean words:",
    round(
        bertopic_dataset["bertopic_input"]
        .str.split()
        .str.len()
        .mean(),
        2
    )
)

Documents: 8469
Empty documents: 0
Mean words: 44.42


In [41]:
url_remaining = bertopic_dataset["bertopic_input"].str.contains(
    r"https?://|www\.",
    case=False,
    regex=True,
    na=False
).sum()

email_remaining = bertopic_dataset["bertopic_input"].str.contains(
    r"\b[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}\b",
    regex=True,
    na=False
).sum()

print("URLs remaining:", url_remaining)
print("Emails remaining:", email_remaining)

URLs remaining: 7
Emails remaining: 0


supportmind_bertopic_cleaned_dataset.csv
              │
              ▼
        bertopic_text
              │
              ▼
     BERTopic preprocessing
       ├─ URLs/emails cleanup
       ├─ whitespace normalization
       ├─ product names preserved
       ├─ issue terminology preserved
       ├─ no stemming
       ├─ no aggressive stopword removal
       └─ no information-destroying cleaning
              │
              ▼
        bertopic_input

In [42]:
import bertopic
import umap
import hdbscan

print("BERTopic:", bertopic.__version__)
print("UMAP:", umap.__version__)
print("HDBSCAN:", hdbscan.__version__ if hasattr(hdbscan, "__version__") else "installed")

c:\Users\danie\anaconda3\envs\supportmind\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BERTopic: 0.17.4
UMAP: 0.5.12
HDBSCAN: installed


In [43]:
from bertopic import BERTopic

# Prepare modelling corpus
documents = (
    bertopic_dataset["bertopic_input"]
    .astype(str)
    .tolist()
)

print(f"Documents for BERTopic: {len(documents):,}")

Documents for BERTopic: 8,469


In [44]:
topic_model = BERTopic(
    verbose=True
)

topics, probabilities = topic_model.fit_transform(documents)

print("\nBERTopic training completed.")
print(f"Documents: {len(topics):,}")
print(f"Unique topic IDs: {len(set(topics))}")

2026-09-13 23:43:56,108 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 265/265 [03:24<00:00,  1.30it/s]
2026-09-13 23:47:31,200 - BERTopic - Embedding - Completed ✓
2026-09-13 23:47:31,201 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-09-13 23:48:18,177 - BERTopic - Dimensionality - Completed ✓
2026-09-13 23:48:18,177 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-09-13 23:48:18,460 - BERTopic - Cluster - Completed ✓
2026-09-13 23:48:18,466 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-09-13 23:48:19,010 - BERTopic - Representation - Completed ✓



BERTopic training completed.
Documents: 8,469
Unique topic IDs: 176


In [45]:
# Inspect discovered topics

topic_info = topic_model.get_topic_info()

print("Number of topics:", len(topic_info) - 1)  # exclude -1 outlier topic

print("\nTopic information:")
display(
    topic_info.head(30)
)

Number of topics: 175

Topic information:


,Topic,Count,Name,Representation,Representative_Docs
0,-1,692,-1_bug_lg_to_in,"[bug, lg, to, in, my, tv, perform, smart, soft...",[My Sony Xperia is making strange noises and n...
1,0,177,0_charge_fitbit_with_having,"[charge, fitbit, with, having, issue, im, an, ...",[I'm having an issue with the Fitbit Charge. I...
2,1,172,1_playstation_sony_having_issue,"[playstation, sony, having, issue, an, product...",[I'm having an issue with the Sony PlayStation...
3,2,169,2_roomba_robot_vacuum_an,"[roomba, robot, vacuum, an, with, issue, havin...",[I'm having an issue with the Roomba Robot Vac...
4,3,167,3_hp_pavilion_issue_having,"[hp, pavilion, issue, having, an, for, the, an...",[I'm having an issue with the HP Pavilion. 1) ...
5,4,163,4_airpods_apple_product_issue,"[airpods, apple, product, issue, having, with,...",[The Apple AirPods is unable to establish a st...
6,5,163,5_versa_smartwatch_fitbit_but,"[versa, smartwatch, fitbit, but, ive, issue, h...",[I'm having an issue with the Fitbit Versa Sma...
7,6,151,6_nikon_having_an_issue,"[nikon, having, an, issue, with, support, you,...",[I'm having an issue with the Nikon D. I'm hav...
8,7,149,7_galaxy_samsung_having_an,"[galaxy, samsung, having, an, null, issue, wit...",[I'm having an issue with the Samsung Galaxy. ...
9,8,144,8_philips_hue_lights_2k,"[philips, hue, lights, 2k, gaming, an, having,...",[I'm having an issue with the Philips Hue Ligh...


In [46]:
# Inspect top words for the largest discovered topics

for topic_id in topic_info[
    topic_info["Topic"] != -1
].head(10)["Topic"]:

    print(f"\nTopic {topic_id}")
    print(topic_model.get_topic(topic_id))


Topic 0
[('charge', np.float64(0.0769716075843485)), ('fitbit', np.float64(0.05972882746172755)), ('with', np.float64(0.006497160146027033)), ('having', np.float64(0.00583766911444087)), ('issue', np.float64(0.005771757951604101)), ('im', np.float64(0.005755521820265863)), ('an', np.float64(0.005630318353422354)), ('the', np.float64(0.0055453558470575405)), ('tried', np.float64(0.005266643843271305)), ('this', np.float64(0.005224490113397775))]

Topic 1
[('playstation', np.float64(0.05306320202852142)), ('sony', np.float64(0.020753514965509777)), ('having', np.float64(0.007241854855300477)), ('issue', np.float64(0.007015902786404935)), ('an', np.float64(0.006876155161590296)), ('product', np.float64(0.006755877525834386)), ('support', np.float64(0.006700777170452235)), ('already', np.float64(0.006367661893997486)), ('with', np.float64(0.006326479212968138)), ('the', np.float64(0.006243746771102185))]

Topic 2
[('roomba', np.float64(0.0717562984044866)), ('robot', np.float64(0.07175629

In [47]:
import re

products = (
    bertopic_dataset["Ticket Description"]
    .dropna()
    .astype(str)
    .unique()
)

# Sort longest first so multi-word product names are matched correctly
products = sorted(products, key=len, reverse=True)

def mask_product_names(text):
    text = str(text)

    for product in products:
        pattern = r"\b" + re.escape(product) + r"\b"
        text = re.sub(
            pattern,
            "PRODUCT",
            text,
            flags=re.IGNORECASE
        )

    return text


bertopic_dataset["bertopic_issue_text"] = (
    bertopic_dataset["bertopic_input"]
    .apply(mask_product_names)
)

print("Documents:", len(bertopic_dataset))
print(
    "Empty documents:",
    (bertopic_dataset["bertopic_issue_text"].str.strip() == "").sum()
)

print("\nExamples:")
display(
    bertopic_dataset[
        ["Ticket Description", "bertopic_issue_text"]
    ].head(10)
)

KeyboardInterrupt: 

In [48]:
import re

# Get the actual product names from the dataset
products = (
    bertopic_dataset
    .merge(
        df[["Ticket ID", "Product Purchased"]],
        on="Ticket ID",
        how="left"
    )["Product Purchased"]
    .dropna()
    .astype(str)
    .unique()
)

# Longest first so multi-word products are matched first
products = sorted(products, key=len, reverse=True)

print("Unique products:", len(products))
print(products)

Unique products: 42
['Nintendo Switch Pro Controller', 'Microsoft Xbox Controller', 'Fitbit Versa Smartwatch', 'Bose SoundLink Speaker', 'Dyson Vacuum Cleaner', 'GoPro Action Camera', 'Roomba Robot Vacuum', 'Philips Hue Lights', 'LG Washing Machine', 'Microsoft Surface', 'Canon DSLR Camera', 'Bose QuietComfort', 'Garmin Forerunner', 'Microsoft Office', 'Autodesk AutoCAD', 'Sony PlayStation', 'Samsung Soundbar', 'Nintendo Switch', 'Nest Thermostat', 'Lenovo ThinkPad', 'Adobe Photoshop', 'Samsung Galaxy', 'Sony 4K HDR TV', 'Amazon Kindle', 'Fitbit Charge', 'Apple AirPods', 'Google Pixel', 'LG Smart TV', 'HP Pavilion', 'Amazon Echo', 'PlayStation', 'Sony Xperia', 'Google Nest', 'MacBook Pro', 'GoPro Hero', 'Canon EOS', 'Dell XPS', 'Asus ROG', 'LG OLED', 'Nikon D', 'iPhone', 'Xbox']


In [49]:
product_pattern = re.compile(
    r"\b(?:"
    + "|".join(re.escape(product) for product in products)
    + r")\b",
    flags=re.IGNORECASE
)

def mask_product_names(text):
    return product_pattern.sub("PRODUCT", str(text))


bertopic_dataset["bertopic_issue_text"] = (
    bertopic_dataset["bertopic_input"]
    .apply(mask_product_names)
)

print("Documents:", len(bertopic_dataset))
print(
    "Empty documents:",
    (bertopic_dataset["bertopic_issue_text"].str.strip() == "").sum()
)

Documents: 8469
Empty documents: 0


In [50]:
display(
    bertopic_dataset[
        ["Ticket ID", "Ticket Description", "bertopic_issue_text"]
    ].head(10)
)

,Ticket ID,Ticket Description,bertopic_issue_text
0,1,I'm having an issue with the GoPro Hero. Pleas...,I'm having an issue with the PRODUCT. Please d...
1,2,I'm having an issue with the LG Smart TV. Plea...,I'm having an issue with the PRODUCT. If you n...
2,3,I'm facing a problem with my Dell XPS. The Del...,I'm facing a problem with my PRODUCT. The PROD...
3,4,I'm having an issue with the Microsoft Office....,I'm having an issue with the PRODUCT. I've alr...
4,5,I'm having an issue with the Autodesk AutoCAD....,I'm having an issue with the PRODUCT. I've not...
5,6,I'm facing a problem with my Microsoft Office....,I'm facing a problem with my PRODUCT. The PROD...
6,7,I'm unable to access my Microsoft Surface acco...,I'm unable to access my PRODUCT account. It ke...
7,8,I'm having an issue with the Philips Hue Light...,I'm having an issue with the PRODUCT. I've per...
8,9,I'm having an issue with the Fitbit Versa Smar...,I'm having an issue with the PRODUCT. PRODUCT ...
9,10,My Dyson Vacuum Cleaner is making strange nois...,My PRODUCT is making strange noises and not fu...


In [51]:
GENERIC_OPENINGS = [
    "i'm having an issue with the product.",
    "i am having an issue with the product.",
    "i'm facing a problem with my product.",
    "i am facing a problem with my product.",
]

for phrase in GENERIC_OPENINGS:
    count = bertopic_dataset["bertopic_issue_text"].str.lower().str.startswith(phrase).sum()
    print(f"{phrase!r}: {count}")

print("\nTop exact opening phrases:")

opening_phrases = (
    bertopic_dataset["bertopic_issue_text"]
    .str.lower()
    .str.extract(r"^(.{0,80}?)(?:\.|!|\?)")[0]
    .value_counts()
    .head(15)
)

display(opening_phrases)

"i'm having an issue with the product.": 5868
'i am having an issue with the product.': 0
"i'm facing a problem with my product.": 179
'i am facing a problem with my product.': 0

Top exact opening phrases:


0
i'm having an issue with the product                                 5868
i'm unable to access my product account                               189
the product is unable to establish a stable internet connection       184
i'm encountering a software bug in the product                        183
my product is making strange noises and not functioning properly      182
i'm facing a problem with my product                                  179
i've noticed a software bug in the product app                        179
i'm having trouble connecting my product to my home wi-fi network     178
there seems to be a hardware problem with my product                  169
i've accidentally deleted important data from my product              165
there seems to be a glitch in the product software                    164
i've encountered a data loss issue with my product                    164
i'm facing issues logging into my product account                     154
my product crashed, and i lost all t

In [52]:
GENERIC_PRODUCT_OPENINGS = {
    "i'm having an issue with the product.",
    "i'm facing a problem with my product.",
}

def remove_generic_product_opening(text):
    text = str(text).strip()

    sentences = re.split(r"(?<=[.!?])\s+", text)

    cleaned = [
        sentence.strip()
        for sentence in sentences
        if sentence.strip().lower() not in GENERIC_PRODUCT_OPENINGS
    ]

    return " ".join(cleaned).strip()


bertopic_dataset["bertopic_issue_text_v2"] = (
    bertopic_dataset["bertopic_issue_text"]
    .apply(remove_generic_product_opening)
)

print("Documents:", len(bertopic_dataset))

print(
    "Empty after Rule 3:",
    (bertopic_dataset["bertopic_issue_text_v2"].str.strip() == "").sum()
)

print(
    "Modified:",
    (
        bertopic_dataset["bertopic_issue_text_v2"]
        != bertopic_dataset["bertopic_issue_text"]
    ).sum()
)

print(
    "Average words:",
    bertopic_dataset["bertopic_issue_text_v2"]
    .str.split()
    .str.len()
    .mean()
)

display(
    bertopic_dataset[
        [
            "Ticket ID",
            "bertopic_issue_text",
            "bertopic_issue_text_v2"
        ]
    ].head(10)
)

Documents: 8469
Empty after Rule 3: 1
Modified: 6044
Average words: 37.18030464045342


,Ticket ID,bertopic_issue_text,bertopic_issue_text_v2
0,1,I'm having an issue with the PRODUCT. Please d...,Please double check your email address. I've t...
1,2,I'm having an issue with the PRODUCT. If you n...,If you need to change an existing product. If ...
2,3,I'm facing a problem with my PRODUCT. The PROD...,The PRODUCT is not turning on. It was working ...
3,4,I'm having an issue with the PRODUCT. I've alr...,I've already contacted customer support multip...
4,5,I'm having an issue with the PRODUCT. I've not...,I've noticed a sudden decrease in battery life...
5,6,I'm facing a problem with my PRODUCT. The PROD...,The PRODUCT is not turning on. It was working ...
6,7,I'm unable to access my PRODUCT account. It ke...,I'm unable to access my PRODUCT account. It ke...
7,8,I'm having an issue with the PRODUCT. I've per...,"I've performed a factory reset on my PRODUCT, ..."
8,9,I'm having an issue with the PRODUCT. PRODUCT ...,PRODUCT is not the exact type you might prefer...
9,10,My PRODUCT is making strange noises and not fu...,My PRODUCT is making strange noises and not fu...


In [53]:
empty_rule3 = bertopic_dataset[
    bertopic_dataset["bertopic_issue_text_v2"].str.strip() == ""
]

display(
    empty_rule3[
        [
            "Ticket ID",
            "Ticket Description",
            "bertopic_issue_text",
            "bertopic_issue_text_v2"
        ]
    ]
)

,Ticket ID,Ticket Description,bertopic_issue_text,bertopic_issue_text_v2
62,63,I'm having an issue with the Xbox. Please assi...,I'm having an issue with the PRODUCT.,


In [54]:
bertopic_training_data = bertopic_dataset[
    bertopic_dataset["bertopic_issue_text_v2"].str.strip() != ""
].copy()

print("Original documents:", len(bertopic_dataset))
print("BERTopic training documents:", len(bertopic_training_data))
print(
    "Excluded documents:",
    len(bertopic_dataset) - len(bertopic_training_data)
)

display(
    bertopic_dataset[
        bertopic_dataset["bertopic_issue_text_v2"].str.strip() == ""
    ][
        ["Ticket ID", "Ticket Description", "bertopic_issue_text_v2"]
    ]
)

Original documents: 8469
BERTopic training documents: 8468
Excluded documents: 1


,Ticket ID,Ticket Description,bertopic_issue_text_v2
62,63,I'm having an issue with the Xbox. Please assi...,


------------------------------modeltrainnig--------------------------

In [55]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=5
)

print("Vectorizer configured")
print("ngram_range:", vectorizer_model.ngram_range)
print("min_df:", vectorizer_model.min_df)
print("stop_words: english")

Vectorizer configured
ngram_range: (1, 2)
min_df: 5
stop_words: english


In [56]:
from bertopic import BERTopic

documents_v2 = (
    bertopic_training_data["bertopic_issue_text_v2"]
    .astype(str)
    .tolist()
)

topic_model_v2 = BERTopic(
    vectorizer_model=vectorizer_model,
    verbose=True
)

topics_v2, probabilities_v2 = topic_model_v2.fit_transform(
    documents_v2
)

print("Documents:", len(documents_v2))
print("Unique topic IDs:", len(set(topics_v2)))
print("Outliers (-1):", sum(topic == -1 for topic in topics_v2))

2026-09-14 00:18:31,228 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 265/265 [01:26<00:00,  3.07it/s]
2026-09-14 00:20:04,913 - BERTopic - Embedding - Completed ✓
2026-09-14 00:20:04,915 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-09-14 00:20:09,627 - BERTopic - Dimensionality - Completed ✓
2026-09-14 00:20:09,627 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-09-14 00:20:09,954 - BERTopic - Cluster - Completed ✓
2026-09-14 00:20:09,961 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-09-14 00:20:10,694 - BERTopic - Representation - Completed ✓


Documents: 8468
Unique topic IDs: 124
Outliers (-1): 1646


In [57]:
topic_info_v2 = topic_model_v2.get_topic_info()

display(
    topic_info_v2[
        ["Topic", "Count", "Name", "Representation"]
    ].head(25)
)

,Topic,Count,Name,Representation
0,-1,1646,-1_ive_checked_ive checked_issue,"[ive, checked, ive checked, issue, product, up..."
1,0,346,0_error message_popping_peculiar error_peculiar,"[error message, popping, peculiar error, pecul..."
2,1,318,1_im concerned_concerned security_security pro...,"[im concerned, concerned security, security pr..."
3,2,312,2_updated firmware_afterward related_afterward...,"[updated firmware, afterward related, afterwar..."
4,3,282,3_update havent_occurring recent_started occur...,"[update havent, occurring recent, started occu..."
5,4,195,4_charger_original charger_product charging_ch...,"[charger, original charger, product charging, ..."
6,5,191,5_password_working_reset_product account,"[password, working, reset, product account, op..."
7,6,189,6_desired action_perform desired_unable option...,"[desired action, perform desired, unable optio..."
8,7,183,7_battery life_sudden decrease_used longer_sudden,"[battery life, sudden decrease, used longer, s..."
9,8,176,8_connection_internet_stable_keeps,"[connection, internet, stable, keeps, unable, ..."


In [58]:
display(
    topic_info_v2[
        topic_info_v2["Topic"] != -1
    ][
        ["Topic", "Count", "Name", "Representation"]
    ]
    .sort_values("Count", ascending=False)
    .head(15)
)

,Topic,Count,Name,Representation
1,0,346,0_error message_popping_peculiar error_peculiar,"[error message, popping, peculiar error, pecul..."
2,1,318,1_im concerned_concerned security_security pro...,"[im concerned, concerned security, security pr..."
3,2,312,2_updated firmware_afterward related_afterward...,"[updated firmware, afterward related, afterwar..."
4,3,282,3_update havent_occurring recent_started occur...,"[update havent, occurring recent, started occu..."
5,4,195,4_charger_original charger_product charging_ch...,"[charger, original charger, product charging, ..."
6,5,191,5_password_working_reset_product account,"[password, working, reset, product account, op..."
7,6,189,6_desired action_perform desired_unable option...,"[desired action, perform desired, unable optio..."
8,7,183,7_battery life_sudden decrease_used longer_sudden,"[battery life, sudden decrease, used longer, s..."
9,8,176,8_connection_internet_stable_keeps,"[connection, internet, stable, keeps, unable, ..."
10,9,175,9_problem product_hardware_screen_product screen,"[problem product, hardware, screen, product sc..."


In [59]:
topic_12_docs = topic_model_v2.get_representative_docs(12)

print("Topic 12 representative documents:")
for i, doc in enumerate(topic_12_docs, 1):
    print(f"\n--- Document {i} ---")
    print(doc)

Topic 12 representative documents:

--- Document 1 ---
#PRODUCT.product " a[0].name = "panther" b[0].name = "peppermint The issue I'm facing is intermittent. Sometimes it works fine, but other times it acts up unexpectedly.

--- Document 2 ---
PRODUCT PRODUCT if (product_name == "Raffle" ) The issue I'm facing is intermittent. Sometimes it works fine, but other times it acts up unexpectedly.

--- Document 3 ---
Product ID # I've followed the troubleshooting steps mentioned in the user manual, but the issue persists.


In [60]:
contamination_patterns = [
    r"#PRODUCT",
    r"productid",
    r"product\.product",
    r"product id",
    r"\bif\s*\(\s*product_name",
    r"\ba\[\d+\]\.name\s*=",
    r"\bb\[\d+\]\.name\s*=",
]

for pattern in contamination_patterns:
    matches = bertopic_dataset["bertopic_issue_text_v2"].str.contains(
        pattern,
        case=False,
        regex=True,
        na=False
    )

    print(f"{pattern:35} → {matches.sum()} documents")

#PRODUCT                            → 13 documents
productid                           → 13 documents
product\.product                    → 4 documents
product id                          → 62 documents
\bif\s*\(\s*product_name            → 2 documents
\ba\[\d+\]\.name\s*=                → 1 documents
\bb\[\d+\]\.name\s*=                → 1 documents


In [61]:
product_id_docs = bertopic_dataset[
    bertopic_dataset["bertopic_issue_text_v2"]
    .str.contains(r"\bproduct id\b", case=False, regex=True, na=False)
]

print("Documents containing 'product id':", len(product_id_docs))

display(
    product_id_docs[
        [
            "Ticket ID",
            "bertopic_issue_text_v2"
        ]
    ].head(20)
)

Documents containing 'product id': 58


,Ticket ID,bertopic_issue_text_v2
77,78,Product #8 4-8-0-1 The product code on your pr...
194,195,(I'm using 'PRODUCT ' as a product id in my co...
494,495,Product ID: 294823 Model: D3-M3 Year: 2015 Gra...
778,779,Product ID: 0 Date of Use: 04/18/2018 format: ...
802,803,Product ID: 0 3 Sold Unavailable I'm not sure ...
878,879,Product ID: 1413882 Product ID: 1413882 Date: ...
1098,1099,"If we missed any of the product details, pleas..."
1126,1127,Product ID # I've followed the troubleshooting...
1154,1155,This is the product ID and it should not be in...
1342,1343,Product ID: 1680 Process: eos Total price: $12...


In [62]:
for _, row in product_id_docs.head(20).iterrows():
    print(f"\n{'=' * 80}")
    print("Ticket ID:", row["Ticket ID"])
    print(row["bertopic_issue_text_v2"])


Ticket ID: 78
Product #8 4-8-0-1 The product code on your product. #Product ID : PRODUCT I've checked the device settings and made sure that everything is configured correctly.

Ticket ID: 195
(I'm using 'PRODUCT ' as a product id in my code to indicate the 'id' I want to share). Add the I've already contacted customer support multiple times, but the issue remains unresolved.

Ticket ID: 495
Product ID: 294823 Model: D3-M3 Year: 2015 Grade: B2 Grade point: I'm experiencing this issue on multiple devices of the same model, so it seems to be a widespread problem.

Ticket ID: 779
Product ID: 0 Date of Use: 04/18/2018 format: wb league: uk playback I've recently updated the firmware of my PRODUCT, and the issue started happening afterward. Could it be related to the update?

Ticket ID: 803
Product ID: 0 3 Sold Unavailable I'm not sure if this issue is specific to my device or if others have reported similar problems.

Ticket ID: 879
Product ID: 1413882 Product ID: 1413882 Date: 11 Mar 201

In [63]:
import re


def remove_confirmed_metadata_fragments(text):
    text = str(text)

    # Code-like fragments
    text = re.sub(
        r"\bif\s*\(\s*product_name\s*==.*?\)",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\b[a-z]\[\d+\]\.name\s*=\s*['\"].*?['\"]",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\b(?:\$purchase|productid)\s*=\s*[^;]+;?",
        " ",
        text,
        flags=re.IGNORECASE
    )

    # Explicit malformed Product-ID markers
    text = re.sub(
        r"#product\s*id\s*:\s*product\b",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\bproduct\s*id\s*#\b",
        " ",
        text,
        flags=re.IGNORECASE
    )

    # Standalone marketplace/cart fragments
    text = re.sub(
        r"\b\d+\s+\d+\s+\d+\s+reviews\s+add\s+to\s+cart\b",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\badd\s+to\s+cart\b",
        " ",
        text,
        flags=re.IGNORECASE
    )

    # Clean spacing
    text = re.sub(r"\s+", " ", text).strip()

    return text


bertopic_dataset["bertopic_issue_text_v3"] = (
    bertopic_dataset["bertopic_issue_text_v2"]
    .apply(remove_confirmed_metadata_fragments)
)

print("Documents:", len(bertopic_dataset))

print(
    "Empty after metadata cleanup:",
    (bertopic_dataset["bertopic_issue_text_v3"].str.strip() == "").sum()
)

print(
    "Modified:",
    (
        bertopic_dataset["bertopic_issue_text_v3"]
        != bertopic_dataset["bertopic_issue_text_v2"]
    ).sum()
)

Documents: 8469
Empty after metadata cleanup: 1
Modified: 25


In [64]:
empty_v3 = bertopic_dataset[
    bertopic_dataset["bertopic_issue_text_v3"].str.strip() == ""
]

display(
    empty_v3[
        [
            "Ticket ID",
            "Ticket Description",
            "bertopic_issue_text_v3"
        ]
    ]
)

,Ticket ID,Ticket Description,bertopic_issue_text_v3
62,63,I'm having an issue with the Xbox. Please assi...,


In [65]:
check_patterns = [
    r"#PRODUCT",
    r"product\.product",
    r"\bif\s*\(\s*product_name",
    r"\ba\[\d+\]\.name\s*=",
    r"\bb\[\d+\]\.name\s*=",
    r"\bproductid\b",
]

for pattern in check_patterns:
    count = bertopic_dataset["bertopic_issue_text_v3"].str.contains(
        pattern,
        case=False,
        regex=True,
        na=False
    ).sum()

    print(f"{pattern:35} → {count}")

#PRODUCT                            → 12
product\.product                    → 4
\bif\s*\(\s*product_name            → 1
\ba\[\d+\]\.name\s*=                → 0
\bb\[\d+\]\.name\s*=                → 0
\bproductid\b                       → 10


In [66]:
remaining_patterns = {
    "#PRODUCT": r"#PRODUCT",
    "product.product": r"product\.product",
    "if product_name": r"\bif\s*\(\s*product_name",
    "productid": r"\bproductid\b",
}

for name, pattern in remaining_patterns.items():

    matches = bertopic_dataset[
        bertopic_dataset["bertopic_issue_text_v3"].str.contains(
            pattern,
            case=False,
            regex=True,
            na=False
        )
    ]

    print(f"\n{'=' * 90}")
    print(f"{name} → {len(matches)} documents")

    for _, row in matches.iterrows():
        print(f"\nTicket ID: {row['Ticket ID']}")
        print(row["bertopic_issue_text_v3"])


#PRODUCT → 12 documents

Ticket ID: 2419
#PRODUCT is the last name that the item has been sold so far.. $this can't be a product I ordered from and I'm missing The issue I'm facing is intermittent. Sometimes it works fine, but other times it acts up unexpectedly.

Ticket ID: 3195
Thanks. #PRODUCT We've received a message claiming to be from an individual here, but we are unable to confirm who I've already contacted customer support multiple times, but the issue remains unresolved.

Ticket ID: 4162
#purchase #id=1422 #product_sell #id=1421 #value@product_purchased #id=1420 #product_ I've reviewed the troubleshooting steps on the official support website, but they didn't resolve the problem.

Ticket ID: 5207
#PRODUCT.product " m facing is intermittent. Sometimes it works fine, but other times it acts up unexpectedly.

Ticket ID: 5901
I'm having an issue with the code. $ ( '#product_purchasement' ). _ -> product_purchasement ([ 1 I've reviewed the troubleshooting steps on the official su

In [67]:
import re


def remove_final_confirmed_contamination(text):
    text = str(text)

    # Code-like product-name condition
    text = re.sub(
        r'if\s*\(\s*product_name\s*==.*?\)',
        ' ',
        text,
        flags=re.IGNORECASE
    )

    # JavaScript / code-style product references
    text = re.sub(
        r"\$\s*\(\s*['\"]#?product[^)]*['\"]\s*\).*?(?=I'm|I’m|The|My|I've|I’ve|This|There|It|I'm|If)",
        " ",
        text,
        flags=re.IGNORECASE
    )

    # Array/object code fragments
    text = re.sub(
        r'\b[a-z]\[\d+\]\.name\s*=\s*["\'].*?["\']',
        ' ',
        text,
        flags=re.IGNORECASE
    )

    # Explicit malformed product markers
    text = re.sub(
        r'#product(?:\.product|[_a-z]*)?',
        ' ',
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r'\bproduct\.product(?:_method)?\s*\([^)]*\)',
        ' ',
        text,
        flags=re.IGNORECASE
    )

    # Code-style ProductID references
    text = re.sub(
        r'\b(?:productid|productId|ProductID)\b',
        ' ',
        text
    )

    # Clean leftover obvious code punctuation
    text = re.sub(
        r'(?<!\w)[\$#]\s*(?=[\(\[])',
        ' ',
        text
    )

    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text


bertopic_dataset["bertopic_issue_text_final"] = (
    bertopic_dataset["bertopic_issue_text_v3"]
    .apply(remove_final_confirmed_contamination)
)

print("Documents:", len(bertopic_dataset))

print(
    "Empty documents:",
    (
        bertopic_dataset["bertopic_issue_text_final"]
        .str.strip() == ""
    ).sum()
)

print(
    "Modified:",
    (
        bertopic_dataset["bertopic_issue_text_final"]
        != bertopic_dataset["bertopic_issue_text_v3"]
    ).sum()
)

Documents: 8469
Empty documents: 1
Modified: 28


In [68]:
patterns = {
    "#PRODUCT": r"#PRODUCT",
    "product.product": r"product\.product",
    "if product_name": r"\bif\s*\(\s*product_name",
    "productID": r"\bproductid\b",
}

for name, pattern in patterns.items():

    count = bertopic_dataset[
        "bertopic_issue_text_final"
    ].str.contains(
        pattern,
        case=False,
        regex=True,
        na=False
    ).sum()

    print(f"{name:25} → {count}")

#PRODUCT                  → 0
product.product           → 2
if product_name           → 1
productID                 → 3


In [69]:
display(
    bertopic_dataset[
        bertopic_dataset["bertopic_issue_text_final"].str.strip() == ""
    ][
        ["Ticket ID", "Ticket Description", "bertopic_issue_text_final"]
    ]
)

,Ticket ID,Ticket Description,bertopic_issue_text_final
62,63,I'm having an issue with the Xbox. Please assi...,


In [70]:
patterns = {
    "product.product": r"product\.product",
    "if product_name": r"\bif\s*\(\s*product_name",
    "productID": r"\bproductid\b",
}

for name, pattern in patterns.items():
    mask = bertopic_dataset["bertopic_issue_text_final"].str.contains(
        pattern,
        case=False,
        regex=True,
        na=False
    )

    print(f"\n{'=' * 70}")
    print(f"{name} → {mask.sum()} matches")
    print("=" * 70)

    display(
        bertopic_dataset.loc[
            mask,
            [
                "Ticket ID",
                "Ticket Description",
                "bertopic_issue_text_v3",
                "bertopic_issue_text_final",
            ]
        ]
    )


product.product → 2 matches


,Ticket ID,Ticket Description,bertopic_issue_text_v3,bertopic_issue_text_final
3445,3446,I'm having an issue with the Apple AirPods. Pl...,It is not a problem with the PRODUCT.product s...,It is not a problem with the PRODUCT.product s...
8046,8047,I'm having an issue with the Autodesk AutoCAD....,PRODUCT is not a valid key type by default PRO...,PRODUCT is not a valid key type by default PRO...



if product_name → 1 matches


,Ticket ID,Ticket Description,bertopic_issue_text_v3,bertopic_issue_text_final
284,285,I'm having an issue with the Microsoft Xbox Co...,""" > > Note: There may be a few of these errors...",""" > > Note: There may be a few of these errors..."



productID → 3 matches


,Ticket ID,Ticket Description,bertopic_issue_text_v3,bertopic_issue_text_final
484,485,I'm having an issue with the Canon DSLR Camera...,if (!productID.length) I've checked for any av...,if (!productID.length) I've checked for any av...
2783,2784,I'm having an issue with the Adobe Photoshop. ...,if ( ProductId. ProductID_OK ) else I'm using ...,if ( ProductId. ProductID_OK ) else I'm using ...
4107,4108,I'm having an issue with the iPhone. Please as...,"( defun myProductWanted ( item :Product, itemI...","( defun myProductWanted ( item :Product, itemI..."


In [71]:
def remove_last_confirmed_contamination(text):
    text = str(text)

    # JavaScript / code-style ProductID checks
    text = re.sub(
        r'if\s*\(\s*!?\s*productid\s*\.?\s*[a-z_]*\s*\)',
        ' ',
        text,
        flags=re.IGNORECASE
    )

    # ProductId/ProductID conditional code
    text = re.sub(
        r'if\s*\(\s*productid\s*\.\s*productid_ok\s*\)\s*else',
        ' ',
        text,
        flags=re.IGNORECASE
    )

    # Lisp/Common-Lisp style product function fragment
    text = re.sub(
        r'\(\s*defun\s+myproductwanted\b.*?\)',
        ' ',
        text,
        flags=re.IGNORECASE
    )

    # Remaining PRODUCT.product contamination
    text = re.sub(
        r'\bproduct\.product\b',
        ' ',
        text,
        flags=re.IGNORECASE
    )

    # Remaining code-style ProductID tokens
    text = re.sub(
        r'\bproductid\b',
        ' ',
        text,
        flags=re.IGNORECASE
    )

    # Clean obvious leftover code punctuation
    text = re.sub(r'\s+', ' ', text).strip()

    return text


bertopic_dataset["bertopic_issue_text_final_v2"] = (
    bertopic_dataset["bertopic_issue_text_final"]
    .apply(remove_last_confirmed_contamination)
)

In [72]:
print("Documents:", len(bertopic_dataset))

print(
    "Empty documents:",
    (
        bertopic_dataset["bertopic_issue_text_final_v2"]
        .str.strip() == ""
    ).sum()
)

print(
    "Modified:",
    (
        bertopic_dataset["bertopic_issue_text_final_v2"]
        != bertopic_dataset["bertopic_issue_text_final"]
    ).sum()
)

Documents: 8469
Empty documents: 1
Modified: 5


In [73]:
patterns = {
    "product.product": r"product\.product",
    "if product_name": r"\bif\s*\(\s*product_name",
    "productID": r"\bproductid\b",
    "defun myProductWanted": r"\bdefun\s+myProductWanted\b",
}

for name, pattern in patterns.items():
    count = bertopic_dataset[
        "bertopic_issue_text_final_v2"
    ].str.contains(
        pattern,
        case=False,
        regex=True,
        na=False
    ).sum()

    print(f"{name:25} → {count}")

product.product           → 0
if product_name           → 1
productID                 → 0
defun myProductWanted     → 0


In [74]:
bertopic_training_data = bertopic_dataset[
    bertopic_dataset["bertopic_issue_text_final_v2"].str.strip() != ""
].copy()

documents = (
    bertopic_training_data["bertopic_issue_text_final_v2"]
    .astype(str)
    .tolist()
)

print("Training documents:", len(documents))

Training documents: 8468


In [75]:
import numpy as np
import pandas as pd

from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

print("BERTopic:", __import__("bertopic").__version__)
print("UMAP:", __import__("umap").__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

BERTopic: 0.17.4
UMAP: 0.5.12
NumPy: 2.4.6
Pandas: 3.0.3


In [76]:
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42,
)

print(umap_model)

UMAP(metric='cosine', min_dist=0.0, n_components=5, random_state=42)


In [77]:
hdbscan_model = HDBSCAN(
    min_cluster_size=30,
    min_samples=10,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True,
)

print(hdbscan_model)

HDBSCAN(min_cluster_size=30, min_samples=10, prediction_data=True)


In [78]:
vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=5,
)

print(vectorizer_model)

CountVectorizer(min_df=5, ngram_range=(1, 2), stop_words='english')


In [79]:
from fastembed import TextEmbedding

embedding_model = TextEmbedding(
    model_name="BAAI/bge-base-en-v1.5"
)

print("Embedding model loaded:")
print(embedding_model.model_name)

Embedding model loaded:
BAAI/bge-base-en-v1.5


In [80]:
topic_model = BERTopic(
    embedding_model=None,          # We will provide our own BGE embeddings
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    verbose=True,
)

print(topic_model)

BERTopic(calculate_probabilities=False, ctfidf_model=ClassTfidfTransformer(...), embedding_model=None, hdbscan_model=HDBSCAN(...), language=english, low_memory=False, min_topic_size=10, n_gram_range=(1, 1), nr_topics=None, representation_model=None, seed_topic_list=None, top_n_words=10, umap_model=UMAP(...), vectorizer_model=CountVectorizer(...), verbose=True, zeroshot_min_similarity=0.7, zeroshot_topic_list=None)


In [81]:
print("Documents:", len(documents))
print("Column:", "bertopic_issue_text_final_v2")

print("\nFirst 3 modelling documents:\n")

for i, text in enumerate(documents[:3], start=1):
    print(f"--- Document {i} ---")
    print(text)
    print()

Documents: 8468
Column: bertopic_issue_text_final_v2

First 3 modelling documents:

--- Document 1 ---
Please double check your email address. I've tried troubleshooting steps mentioned in the user manual, but the issue persists.

--- Document 2 ---
If you need to change an existing product. If The issue I'm facing is intermittent. Sometimes it works fine, but other times it acts up unexpectedly.

--- Document 3 ---
The PRODUCT is not turning on. It was working fine until yesterday, but now it doesn't respond. 1.8.3 I really I'm using the original charger that came with my PRODUCT, but it's not charging properly.



In [82]:
# Inspect the shortest modelling documents.
lengths = (
    bertopic_training_data["bertopic_issue_text_final_v2"]
    .astype(str)
    .str.split()
    .str.len()
)

shortest = bertopic_training_data.loc[
    lengths.nsmallest(30).index,
    [
        "Ticket ID",
        "Ticket Description",
        "bertopic_issue_text_final_v2",
    ],
].copy()

display(shortest)

,Ticket ID,Ticket Description,bertopic_issue_text_final_v2
728,729,I'm having an issue with the LG Washing Machin...,<Exclude
7200,7201,I'm having an issue with the iPhone. Please as...,;
8331,8332,I'm having an issue with the Microsoft Xbox Co...,""""
790,791,I'm having an issue with the LG OLED. Please a...,""" )"
4601,4602,I'm having an issue with the Apple AirPods. Pl...,Note: This
6440,6441,I'm having an issue with the Roomba Robot Vacu...,## var
1169,1170,I'm having an issue with the Canon EOS. Please...,"PRODUCT.add to , ("
7034,7035,I'm having an issue with the Asus ROG. Please ...,I appreciate your time. return export
3,4,I'm having an issue with the Microsoft Office....,I've already contacted customer support multip...
1242,1243,I'm having an issue with the HP Pavilion. Plea...,I've already contacted customer support multip...


In [83]:
print("Word-count statistics:")
print(lengths.describe())

print("\nDocuments with <= 10 words:", (lengths <= 10).sum())
print("Documents with <= 15 words:", (lengths <= 15).sum())
print("Documents with <= 20 words:", (lengths <= 20).sum())

Word-count statistics:
count    8468.000000
mean       37.163321
std         9.530127
min         1.000000
25%        33.000000
50%        40.000000
75%        44.000000
max        61.000000
Name: bertopic_issue_text_final_v2, dtype: float64

Documents with <= 10 words: 8
Documents with <= 15 words: 290
Documents with <= 20 words: 754


In [84]:
short_mask = (
    bertopic_training_data["bertopic_issue_text_final_v2"]
    .astype(str)
    .str.split()
    .str.len()
    <= 10
)

display(
    bertopic_training_data.loc[
        short_mask,
        [
            "Ticket ID",
            "Ticket Subject",
            "Ticket Description",
            "bertopic_issue_text_final_v2",
        ],
    ]
)

,Ticket ID,Ticket Subject,Ticket Description,bertopic_issue_text_final_v2
728,729,Delivery problem,I'm having an issue with the LG Washing Machin...,<Exclude
790,791,Product recommendation,I'm having an issue with the LG OLED. Please a...,""" )"
1169,1170,Battery life,I'm having an issue with the Canon EOS. Please...,"PRODUCT.add to , ("
4601,4602,Peripheral compatibility,I'm having an issue with the Apple AirPods. Pl...,Note: This
6440,6441,Cancellation request,I'm having an issue with the Roomba Robot Vacu...,## var
7034,7035,Product setup,I'm having an issue with the Asus ROG. Please ...,I appreciate your time. return export
7200,7201,Product compatibility,I'm having an issue with the iPhone. Please as...,;
8331,8332,Cancellation request,I'm having an issue with the Microsoft Xbox Co...,""""


In [85]:
INVALID_BERTOPIC_TICKETS = {
    63,
    728,
    790,
    1169,
    4601,
    6440,
    7034,
    7200,
    8331,
}

bertopic_training_data_final = bertopic_dataset[
    ~bertopic_dataset["Ticket ID"].isin(INVALID_BERTOPIC_TICKETS)
].copy()

documents_final = (
    bertopic_training_data_final[
        "bertopic_issue_text_final_v2"
    ]
    .astype(str)
    .tolist()
)

print("Original documents:", len(bertopic_dataset))
print("Excluded documents:", len(INVALID_BERTOPIC_TICKETS))
print("Final BERTopic documents:", len(documents_final))
print(
    "Empty documents:",
    (
        bertopic_training_data_final[
            "bertopic_issue_text_final_v2"
        ].str.strip() == ""
    ).sum()
)

Original documents: 8469
Excluded documents: 9
Final BERTopic documents: 8460
Empty documents: 0


In [86]:
embeddings = np.array(
    list(
        embedding_model.embed(
            documents_final
        )
    )
)

print("Embedding shape:", embeddings.shape)
print("Embedding dtype:", embeddings.dtype)
print("Expected documents:", len(documents_final))

Embedding shape: (8460, 768)
Embedding dtype: float32
Expected documents: 8460


In [87]:
from pathlib import Path

embedding_path = Path(
    r"C:\Users\danie\supportmind-ai\data\processed\supportmind_bertopic_embeddings.npy"
)

np.save(embedding_path, embeddings)

print("Saved:", embedding_path)
print("File exists:", embedding_path.exists())
print("Shape:", np.load(embedding_path, mmap_mode="r").shape)

Saved: C:\Users\danie\supportmind-ai\data\processed\supportmind_bertopic_embeddings.npy
File exists: True
Shape: (8460, 768)


In [88]:
topics, probabilities = topic_model.fit_transform(
    documents_final,
    embeddings=embeddings,
)

print("Training complete.")
print("Documents:", len(topics))
print("Unique topic IDs:", len(set(topics)))
print("Outliers (-1):", sum(t == -1 for t in topics))

2026-09-14 01:21:15,922 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-09-14 01:21:28,724 - BERTopic - Dimensionality - Completed ✓
2026-09-14 01:21:28,724 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-09-14 01:21:28,876 - BERTopic - Cluster - Completed ✓
2026-09-14 01:21:28,879 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-09-14 01:21:29,168 - BERTopic - Representation - Completed ✓


Training complete.
Documents: 8460
Unique topic IDs: 60
Outliers (-1): 1379


In [89]:
topic_info = topic_model.get_topic_info()

display(
    topic_info[
        ["Topic", "Count", "Name"]
    ].head(20)
)

,Topic,Count,Name
0,-1,1379,-1_issue_problem_device_ive
1,0,393,0_productivity_need assistance_assistance soon...
2,1,349,1_error message_screen says_popping_popping pr...
3,2,335,2_updated firmware_happening afterward_afterwa...
4,3,318,3_software updates_updates product_updates_ive...
5,4,306,4_acts unexpectedly_acts_intermittent works_fi...
6,5,294,5_action product_option perform_perform desire...
7,6,273,6_im concerned_security product_like ensure_en...
8,7,229,7_errors_app_product app_loss
9,8,228,8_used longer_sudden_sudden decrease_noticed s...


In [90]:
print("Total topics excluding outlier:", (topic_info["Topic"] != -1).sum())
print("Outlier count:", topic_info.loc[topic_info["Topic"] == -1, "Count"].iloc[0])

print("\nTopic size statistics:")
print(
    topic_info.loc[
        topic_info["Topic"] != -1,
        "Count"
    ].describe()
)

Total topics excluding outlier: 59
Outlier count: 1379

Topic size statistics:
count     59.000000
mean     120.016949
std       94.198706
min       31.000000
25%       46.000000
50%       77.000000
75%      170.000000
max      393.000000
Name: Count, dtype: float64


In [91]:
for topic_id in topic_info.loc[
    topic_info["Topic"] != -1,
    "Topic"
].head(30):

    print(f"\n{'=' * 70}")
    print(f"TOPIC {topic_id}")
    print("=" * 70)

    words = topic_model.get_topic(topic_id)

    print(
        ", ".join(
            f"{word} ({score:.3f})"
            for word, score in words[:10]
        )
    )


TOPIC 0
productivity (0.085), need assistance (0.063), assistance soon (0.062), affecting work (0.062), possible affecting (0.062), work productivity (0.062), affecting (0.062), soon possible (0.062), assistance (0.061), soon (0.060)

TOPIC 1
error message (0.077), screen says (0.077), popping (0.077), popping product (0.077), noticed peculiar (0.077), message popping (0.077), peculiar (0.077), peculiar error (0.077), message (0.071), product screen (0.067)

TOPIC 2
updated firmware (0.072), happening afterward (0.072), afterward (0.072), afterward related (0.072), started happening (0.072), related update (0.072), firmware product (0.072), issue started (0.072), recently updated (0.072), firmware (0.072)

TOPIC 3
software updates (0.098), updates product (0.097), updates (0.094), ive checked (0.082), checked (0.082), checked software (0.066), running latest (0.066), product running (0.066), latest version (0.065), latest (0.063)

TOPIC 4
acts unexpectedly (0.080), acts (0.080), inter

In [92]:
for topic_id in range(30):

    if topic_id not in topic_model.get_topics():
        continue

    print(f"\n{'=' * 80}")
    print(f"TOPIC {topic_id}")
    print("=" * 80)

    docs = topic_model.get_representative_docs(topic_id)

    for i, doc in enumerate(docs[:5], start=1):
        print(f"\n{i}. {doc}")


TOPIC 0

1. 1.4.9 [3DS] [NieR:Automata] NieR:Automata [3DS] I need assistance as soon as possible because it's affecting my work and productivity.

2. minutes_to_update?app_id=152634 I need assistance as soon as possible because it's affecting my work and productivity.

3. PRODUCT <div I need assistance as soon as possible because it's affecting my work and productivity.

TOPIC 1

1. _______________________________________________ I've noticed a peculiar error message popping up on my PRODUCT screen. It says ''.

2. I've noticed a peculiar error message popping up on my PRODUCT screen. It says ''.

3. I've noticed a peculiar error message popping up on my PRODUCT screen. It says ''.

TOPIC 2

1. I've recently updated the firmware of my PRODUCT, and the issue started happening afterward. Could it be related to the update?

2. 0x0000000000000 0x00000000000000 0x00000000000000 0x00000000000000 0x00000000000000 0 I've recently updated the firmware of my PRODUCT, and the issue started happ

In [93]:
topic_info = topic_model.get_topic_info()

topic_sizes = (
    topic_info[
        topic_info["Topic"] != -1
    ]["Count"]
)

print("Number of topics:", len(topic_sizes))
print("Minimum topic size:", topic_sizes.min())
print("Median topic size:", topic_sizes.median())
print("Mean topic size:", topic_sizes.mean())
print("Maximum topic size:", topic_sizes.max())

print("\nSmall topics:")
print((topic_sizes < 30).sum())

print("\nTopics with 30+ documents:")
print((topic_sizes >= 30).sum())

Number of topics: 59
Minimum topic size: 31
Median topic size: 77.0
Mean topic size: 120.01694915254237
Maximum topic size: 393

Small topics:
0

Topics with 30+ documents:
59


In [95]:
topic_assignments = topic_model.topics_

outlier_count = sum(topic == -1 for topic in topic_assignments)
total_documents = len(topic_assignments)

outlier_rate = outlier_count / total_documents * 100
assigned_count = total_documents - outlier_count

print("Total documents:", total_documents)
print("Outlier documents:", outlier_count)
print(f"Outlier rate: {outlier_rate:.2f}%")
print("Assigned to topics:", assigned_count)
print(f"Assigned rate: {assigned_count / total_documents * 100:.2f}%")

Total documents: 8460
Outlier documents: 1379
Outlier rate: 16.30%
Assigned to topics: 7081
Assigned rate: 83.70%


In [96]:
id="r4k8pn"
topic_info = topic_model.get_topic_info()

topic_words = []

for topic_id in topic_info["Topic"]:
    if topic_id == -1:
        continue

    words = [
        word
        for word, score in topic_model.get_topic(topic_id)[:10]
    ]

    topic_words.extend(words)

unique_words = set(topic_words)

topic_diversity = len(unique_words) / len(topic_words)

print("Topics evaluated:", len(topic_words) // 10)
print("Top words per topic: 10")
print("Total topic words:", len(topic_words))
print("Unique topic words:", len(unique_words))
print(f"Topic diversity: {topic_diversity:.4f}")

Topics evaluated: 59
Top words per topic: 10
Total topic words: 590
Unique topic words: 353
Topic diversity: 0.5983


In [98]:
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel

# Tokenize the same corpus used for BERTopic
tokenized_documents = [
    doc.lower().split()
    for doc in documents_final
]

# Build dictionary
dictionary = Dictionary(tokenized_documents)

# Get top words for each discovered topic
topic_words_for_coherence = []

for topic_id in topic_model.get_topics():
    if topic_id == -1:
        continue

    words = [
        word
        for word, score in topic_model.get_topic(topic_id)[:10]
    ]

    topic_words_for_coherence.append(words)

# Calculate C_V coherence
coherence_model = CoherenceModel(
    topics=topic_words_for_coherence,
    texts=tokenized_documents,
    dictionary=dictionary,
    coherence="c_v",
)

coherence_score = coherence_model.get_coherence()

print(f"Number of topics evaluated: {len(topic_words_for_coherence)}")
print(f"Coherence (C_V): {coherence_score:.4f}")

Number of topics evaluated: 59
Coherence (C_V): 0.7972


In [100]:
# Step 1 — Create topic assignments

topic_results = bertopic_training_data_final[
    [
        "Ticket ID",
        "Ticket Subject",
        "bertopic_issue_text_final_v2",
    ]
].copy()

# Add product metadata from the original dataset
topic_results = topic_results.merge(
    df[
        [
            "Ticket ID",
            "Product Purchased",
        ]
    ],
    on="Ticket ID",
    how="left",
    validate="one_to_one",
)

# Attach BERTopic assignments
topic_results["Topic ID"] = topics

print("Rows:", len(topic_results))
print("Columns:", topic_results.columns.tolist())

print("\nMissing product values:")
print(topic_results["Product Purchased"].isna().sum())

print("\nTopic assignment counts:")
print(topic_results["Topic ID"].value_counts().head(15))

Rows: 8460
Columns: ['Ticket ID', 'Ticket Subject', 'bertopic_issue_text_final_v2', 'Product Purchased', 'Topic ID']

Missing product values:
0

Topic assignment counts:
Topic ID
-1     1379
 0      393
 1      349
 2      335
 3      318
 4      306
 5      294
 6      273
 7      229
 8      228
 9      211
 10     193
 11     182
 12     179
 13     175
Name: count, dtype: int64


In [101]:
# Step 2 — Add topic metadata

topic_info = topic_model.get_topic_info()

topic_metadata = topic_info[
    [
        "Topic",
        "Count",
        "Name",
    ]
].copy()

topic_metadata = topic_metadata.rename(
    columns={
        "Topic": "Topic ID",
        "Count": "Topic Size",
        "Name": "Topic Name",
    }
)

topic_results = topic_results.merge(
    topic_metadata,
    on="Topic ID",
    how="left",
    validate="many_to_one",
)

print("Rows:", len(topic_results))
print("Columns:", topic_results.columns.tolist())

print("\nMissing topic metadata:")
print(topic_results[["Topic Size", "Topic Name"]].isna().sum())

print("\nSample:")
print(
    topic_results[
        [
            "Ticket ID",
            "Product Purchased",
            "Topic ID",
            "Topic Size",
            "Topic Name",
        ]
    ].head(10)
)

Rows: 8460
Columns: ['Ticket ID', 'Ticket Subject', 'bertopic_issue_text_final_v2', 'Product Purchased', 'Topic ID', 'Topic Size', 'Topic Name']

Missing topic metadata:
Topic Size    0
Topic Name    0
dtype: int64

Sample:
   Ticket ID        Product Purchased  Topic ID  Topic Size  \
0          1               GoPro Hero        31          71   
1          2              LG Smart TV         4         306   
2          3                 Dell XPS        14         171   
3          4         Microsoft Office        40          48   
4          5         Autodesk AutoCAD         8         228   
5          6         Microsoft Office        14         171   
6          7        Microsoft Surface        33          70   
7          8       Philips Hue Lights        47          44   
8          9  Fitbit Versa Smartwatch         2         335   
9         10     Dyson Vacuum Cleaner        -1        1379   

                                          Topic Name  
0  31_steps mentioned_menti

In [102]:
# Step 3 — Extract topic keywords

topic_keywords = []

for topic_id in topic_model.get_topics():

    if topic_id == -1:
        continue

    words_with_scores = topic_model.get_topic(topic_id)

    for rank, (word, score) in enumerate(
        words_with_scores[:10],
        start=1,
    ):
        topic_keywords.append(
            {
                "Topic ID": topic_id,
                "Rank": rank,
                "Word": word,
                "Score": score,
            }
        )

topic_keywords_df = pd.DataFrame(topic_keywords)

print("Rows:", len(topic_keywords_df))
print("Topics:", topic_keywords_df["Topic ID"].nunique())
print("Keywords per topic:",
      topic_keywords_df.groupby("Topic ID").size().unique())

print("\nSample:")
print(topic_keywords_df.head(20))

Rows: 590
Topics: 59
Keywords per topic: [10]

Sample:
    Topic ID  Rank                Word     Score
0          0     1        productivity  0.084889
1          0     2     need assistance  0.062560
2          0     3     assistance soon  0.062461
3          0     4      affecting work  0.062461
4          0     5  possible affecting  0.062461
5          0     6   work productivity  0.062461
6          0     7           affecting  0.062409
7          0     8       soon possible  0.061639
8          0     9          assistance  0.061389
9          0    10                soon  0.059863
10         1     1       error message  0.076980
11         1     2         screen says  0.076740
12         1     3             popping  0.076740
13         1     4     popping product  0.076740
14         1     5    noticed peculiar  0.076740
15         1     6     message popping  0.076740
16         1     7            peculiar  0.076740
17         1     8      peculiar error  0.076740
18         1  

In [103]:
# Step 4 — Create compact topic summary

topic_summary = (
    topic_keywords_df
    .sort_values(["Topic ID", "Rank"])
    .groupby("Topic ID")
    .agg(
        Top_Keywords=("Word", lambda x: ", ".join(x))
    )
    .reset_index()
)

topic_summary = topic_summary.merge(
    topic_metadata[
        ["Topic ID", "Topic Size", "Topic Name"]
    ],
    on="Topic ID",
    how="left",
    validate="one_to_one",
)

topic_summary = topic_summary[
    [
        "Topic ID",
        "Topic Size",
        "Topic Name",
        "Top_Keywords",
    ]
].sort_values("Topic ID")

print("Topics:", len(topic_summary))
print("Columns:", topic_summary.columns.tolist())

print("\nSample:")
print(topic_summary.head(10).to_string(index=False))

Topics: 59
Columns: ['Topic ID', 'Topic Size', 'Topic Name', 'Top_Keywords']

Sample:
 Topic ID  Topic Size                                                         Topic Name                                                                                                                                                        Top_Keywords
        0         393      0_productivity_need assistance_assistance soon_affecting work                   productivity, need assistance, assistance soon, affecting work, possible affecting, work productivity, affecting, soon possible, assistance, soon
        1         349                1_error message_screen says_popping_popping product                          error message, screen says, popping, popping product, noticed peculiar, message popping, peculiar, peculiar error, message, product screen
        2         335 2_updated firmware_happening afterward_afterward_afterward related updated firmware, happening afterward, afterward, afterward relate

In [104]:
from pathlib import Path

# Step 5 — Save BERTopic outputs

output_dir = Path(
    r"C:\Users\danie\supportmind-ai\data\processed\bertopic"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

topic_results_path = output_dir / "bertopic_ticket_topic_assignments.csv"
topic_summary_path = output_dir / "bertopic_topic_summary.csv"
topic_keywords_path = output_dir / "bertopic_topic_keywords.csv"

topic_results.to_csv(
    topic_results_path,
    index=False,
)

topic_summary.to_csv(
    topic_summary_path,
    index=False,
)

topic_keywords_df.to_csv(
    topic_keywords_path,
    index=False,
)

print("Saved:")
print(topic_results_path)
print(topic_summary_path)
print(topic_keywords_path)

print("\nFile existence:")
print("Ticket assignments:", topic_results_path.exists())
print("Topic summary:", topic_summary_path.exists())
print("Topic keywords:", topic_keywords_path.exists())

Saved:
C:\Users\danie\supportmind-ai\data\processed\bertopic\bertopic_ticket_topic_assignments.csv
C:\Users\danie\supportmind-ai\data\processed\bertopic\bertopic_topic_summary.csv
C:\Users\danie\supportmind-ai\data\processed\bertopic\bertopic_topic_keywords.csv

File existence:
Ticket assignments: True
Topic summary: True
Topic keywords: True


In [105]:
from pathlib import Path

# Step 6 — Prepare production model directory

model_dir = Path(
    r"C:\Users\danie\supportmind-ai\backend\app\nlp\topic_modeling\models\supportmind_bertopic"
)

model_dir.mkdir(
    parents=True,
    exist_ok=True,
)

print("Model directory:")
print(model_dir)

print("\nDirectory exists:", model_dir.exists())
print("Is directory:", model_dir.is_dir())

Model directory:
C:\Users\danie\supportmind-ai\backend\app\nlp\topic_modeling\models\supportmind_bertopic

Directory exists: True
Is directory: True


In [106]:
# Step 6b — Save the trained BERTopic model

topic_model.save(
    str(model_dir),
    serialization="safetensors",
    save_ctfidf=True,
)

print("BERTopic model saved successfully.")
print("Model path:")
print(model_dir)

BERTopic model saved successfully.
Model path:
C:\Users\danie\supportmind-ai\backend\app\nlp\topic_modeling\models\supportmind_bertopic


In [107]:
from bertopic import BERTopic

# Step 7 — Reload the persisted BERTopic model

loaded_topic_model = BERTopic.load(
    str(model_dir)
)

print("Model reloaded successfully.")

print("\nNumber of topics:")
print(len(loaded_topic_model.get_topic_info()) - 1)

print("\nTopic 1:")
print(loaded_topic_model.get_topic(1)[:10])

2026-09-14 02:01:31,730 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.


Model reloaded successfully.

Number of topics:
59

Topic 1:
[['error message', 0.07697975308529437], ['screen says', 0.07673994564065258], ['popping', 0.07673994564065258], ['popping product', 0.07673994564065258], ['noticed peculiar', 0.07673994564065258], ['message popping', 0.07673994564065258], ['peculiar', 0.07673994564065258], ['peculiar error', 0.07673994564065258], ['message', 0.07125273849138684], ['product screen', 0.06744600748972363]]


In [108]:
# Step 8 — Validate inference using the reloaded model

test_documents = documents_final[:5]

inferred_topics, inferred_probabilities = (
    loaded_topic_model.transform(
        test_documents,
        embeddings=embeddings[:5],
    )
)

for i, (document, topic_id) in enumerate(
    zip(test_documents, inferred_topics),
    start=1,
):
    print(f"\n{'=' * 70}")
    print(f"TEST DOCUMENT {i}")
    print(f"{'=' * 70}")
    print("Topic ID:", topic_id)
    print("Text:", document[:300])

2026-09-14 02:02:36,841 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.



TEST DOCUMENT 1
Topic ID: 31
Text: Please double check your email address. I've tried troubleshooting steps mentioned in the user manual, but the issue persists.

TEST DOCUMENT 2
Topic ID: 4
Text: If you need to change an existing product. If The issue I'm facing is intermittent. Sometimes it works fine, but other times it acts up unexpectedly.

TEST DOCUMENT 3
Topic ID: 14
Text: The PRODUCT is not turning on. It was working fine until yesterday, but now it doesn't respond. 1.8.3 I really I'm using the original charger that came with my PRODUCT, but it's not charging properly.

TEST DOCUMENT 4
Topic ID: 40
Text: I've already contacted customer support multiple times, but the issue remains unresolved.

TEST DOCUMENT 5
Topic ID: 8
Text: I've noticed a sudden decrease in battery life on my PRODUCT. It used to last much longer.


🎯 BERTopic is now DONE

We have completed the entire target-driven workflow:

Final Clean Dataset
        ↓
Targeted EDA
        ↓
Targeted preprocessing
        ↓
Product masking
        ↓
BERTopic training corpus
        ↓
BERTopic model
        ↓
Evaluation
        ↓
59 useful topics
        ↓
Production artifacts
        ↓
Save
        ↓
Reload
        ↓
Inference validation ✅
Locked model

BERTopic 0.17.4

Embeddings: BGE-base-en-v1.5 / 768D
UMAP: 15 neighbors, 5 components, cosine
HDBSCAN: min cluster size 30, min samples 10
Vectorizer: English stop words, 1–2 grams, min_df=5
Training documents: 8,460
Topics: 59
Topic coverage: 83.70%
Outliers: 16.30%
Topic diversity: 0.5983
C_V coherence: 0.7972
Saved artifacts
data/processed/
├── supportmind_bertopic_cleaned_dataset.csv
├── supportmind_bertopic_embeddings.npy
└── bertopic/
    ├── bertopic_ticket_topic_assignments.csv
    ├── bertopic_topic_summary.csv
    └── bertopic_topic_keywords.csv

Model:

backend/app/nlp/topic_modeling/models/
└── supportmind_bertopic/

No further tuning is required. We have a model that fits its actual SupportMind role.

The next NLP/ML work should therefore start with the next dataset/task, while keeping the same rule: choose the model because the dataset provides a valid training signal and the model has a real role in SupportMind—not because we want more models.